<a href="https://colab.research.google.com/github/M-Khalid16/PhotonicsAILab_projects/blob/main/Optical_communication_Equalizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Metadata

Domain: Electrical Engineering

Subdomain: - Optical Communication

Difficulty: - Hard

Problem ID: equalizer-deployment-efficiency-audit

References: https://opg.optica.org/oe/fulltext.cfm?uri=oe-34-9-17195

Proof (AV Link after success)-

# Title

Real-World C+L-Band Bayesian Neural Equalizer Engineering Audit

# Main Problem


## Prompt

You are given a CSV file named `cnn_bilstm_equalizer.csv` containing channel/window observations from a coherent optical transmission experiment using a Bayesian CNN-BiLSTM neural equalizer.

The real-world problem is:

A long-haul optical-network engineer needs to decide whether an uncertainty-aware Bayesian CNN-BiLSTM equalizer meaningfully improves C+L-band coherent transmission, how much usable capacity it enables, which wavelength or band regions remain limiting, and whether the gain is worth the computational burden of deployment.

This task combines receiver-performance analysis, C+L-band capacity planning, and deployment-feasibility analysis into one practical engineering report. It is based on the type of problem addressed by long-haul high-capacity C+L-band transmission experiments, where neural equalization is used to improve post-equalization BER, GMI, NGMI, and capacity.

Implement the following function:

```python
def solve_cl_band_equalizer_engineering_audit(
    input_path="/workspace/data/cnn_bilstm_equalizer.csv",
    output_path="/workspace/output.json"
):
```


The solution must read the CSV from `input_path`, clean it deterministically, use all cleaned rows, preserve cleaned row order, and write the final JSON report to `output_path`.

The input file must be read as UTF-8 text. If a Byte Order Mark is present, it must be removed. Line endings must be normalized before CSV parsing. Fully empty rows and fully empty columns must be removed. Column names and string-valued cells must be stripped of surrounding whitespace.

After cleaning, the dataset must contain these required columns:

- `sample_id`
- `channel_index`
- `channel_in_band`
- `band`
- `wavelength_nm`
- `distance_km`
- `loop_count`
- `symbol_rate_GBd`
- `modulation`
- `shaping_entropy_bits`
- `window_length`
- `launch_power_dBm`
- `osnr_dB`
- `srs_tilt_dB`
- `nonlinear_phase_proxy`
- `isi_memory_proxy`
- `amplitude_kurtosis_proxy`
- `center_rx_I`
- `center_rx_Q`
- `pre_context_I`
- `pre_context_Q`
- `post_context_I`
- `post_context_Q`
- `window_power_mean`
- `window_power_std`
- `window_iq_corr`
- `lag1_autocorr_I`
- `lag1_autocorr_Q`
- `target_residual_I`
- `target_residual_Q`
- `target_log_variance`
- `uncertainty_score`
- `ber_wo_nlc`
- `ber_bayesian_cnn_bilstm`
- `gmi_wo_nlc_bits_per_symbol`
- `gmi_bayesian_bits_per_symbol`
- `ngmi_wo_nlc`
- `ngmi_bayesian`
- `capacity_wo_nlc_Gbps`
- `capacity_bayesian_Gbps`
- `capacity_gain_pct`
- `epochs_bayesian_cnn_bilstm`
- `train_split`

Additional columns may be ignored. The `sample_id` field must be non-empty and unique after trimming. The `band` field must contain only `C` or `L`. The `modulation` field must be non-empty. The `train_split` field must be preserved in row-level output but must not be used to remove rows or restrict computation.

All required numeric columns must be finite. The fields `channel_index`, `channel_in_band`, `loop_count`, `symbol_rate_GBd`, `window_length`, and `epochs_bayesian_cnn_bilstm` must be positive. The fields `distance_km`, `shaping_entropy_bits`, `window_power_mean`, `window_power_std`, `uncertainty_score`, `ber_wo_nlc`, `ber_bayesian_cnn_bilstm`, `gmi_wo_nlc_bits_per_symbol`, `gmi_bayesian_bits_per_symbol`, `ngmi_wo_nlc`, `ngmi_bayesian`, `capacity_wo_nlc_Gbps`, and `capacity_bayesian_Gbps` must be nonnegative. The BER values must lie in $ [0,1] $. The NGMI values must lie in $ [0,1] $. The correlation columns `window_iq_corr`, `lag1_autocorr_I`, and `lag1_autocorr_Q` must lie in $ [-1,1] $. The `window_length` value must be odd for every row.

For every row $ i $, compute the equalizer improvement relative to the no-NLC baseline:

$ \Delta \mathrm{BER}_i = \mathrm{BER}^{wo}_i - \mathrm{BER}^{bayes}_i $

$ \Delta \mathrm{GMI}_i = \mathrm{GMI}^{bayes}_i - \mathrm{GMI}^{wo}_i $

$ \Delta \mathrm{NGMI}_i = \mathrm{NGMI}^{bayes}_i - \mathrm{NGMI}^{wo}_i $

$ \Delta C_i = C^{bayes}_i - C^{wo}_i $

Also compute the target residual magnitude:

$ \rho_i = \sqrt{(\Delta I_i)^2 + (\Delta Q_i)^2} $

For capacity planning, classify each row into one channel-quality label:

- `usable` if `ber_bayesian_cnn_bilstm <= 0.05` and `ngmi_bayesian >= 0.80`
- `unusable` if `ber_bayesian_cnn_bilstm > 0.10` or `ngmi_bayesian < 0.70`
- `marginal` otherwise

For deployment feasibility, estimate a deterministic deployment burden from `window_length`, `epochs_bayesian_cnn_bilstm`, `symbol_rate_GBd`, `isi_memory_proxy`, `nonlinear_phase_proxy`, `window_power_std`, `uncertainty_score`, and `target_log_variance`. Estimate equalizer benefit from BER reduction, GMI gain, NGMI gain, capacity gain, and `capacity_gain_pct`. A row is deployment-favorable if it has positive capacity gain, positive GMI gain, positive NGMI gain, and deployment burden below or equal to the dataset median burden.

The solution must use PyTorch for the core numerical calculations. PyTorch tensor operations must operate on data derived from the cleaned CSV and must contribute numerically to the improvement metrics, capacity totals, quality labels, deployment burden, equalizer benefit, ranking, or aggregate summaries. Merely importing PyTorch or returning a boolean flag is insufficient. PyTorch provides deterministic algorithm controls through `torch.use_deterministic_algorithms`, which requires operations to use deterministic algorithms when available for the same input and software/hardware environment.

The solution must be deterministic. Any scoring, ranking, aggregation, or normalization must use fixed deterministic rules and no randomness. Non-finite generated outputs are invalid.

**Requirements:**

The solution must produce a real-world engineering report that answers:

- How much did the Bayesian CNN-BiLSTM equalizer improve the link relative to the no-NLC baseline?
- How much total C+L-band Bayesian-equalized capacity is available?
- Which rows/channels are usable, marginal, or unusable after equalization?
- Which band, wavelength, or channel region is the weakest bottleneck?
- Under what conditions does the equalizer help most?
- Is the equalizer deployment-favorable when benefit is compared with computational burden?

The output must include:

- row-preserving `sample_ids`
- row-level improvement values
- row-level channel-quality labels
- row-level deployment burden, benefit, and deployment-favorable flags
- global capacity and equalizer-improvement summary
- C-band and L-band summaries for all present bands
- bottleneck sample identifiers
- a concise engineering conclusion

**Output JSON**:

The final result must be written to `/workspace/output.json` and returned as a Python dictionary with this structure:

```json
{
  "sample_ids": ["string"],
  "global_summary": {
    "row_count": "int",
    "total_bayesian_capacity_Gbps": "float",
    "total_no_nlc_capacity_Gbps": "float",
    "total_capacity_gain_Gbps": "float",
    "mean_capacity_gain_Gbps": "float",
    "mean_capacity_gain_pct": "float",
    "mean_ber_reduction": "float",
    "mean_gmi_gain": "float",
    "mean_ngmi_gain": "float",
    "usable_channel_count": "int",
    "marginal_channel_count": "int",
    "unusable_channel_count": "int",
    "deployment_favorable_count": "int",
    "deployment_favorable_fraction": "float"
  },
  "band_summary": {
    "band": {
      "row_count": "int",
      "total_bayesian_capacity_Gbps": "float",
      "total_capacity_gain_Gbps": "float",
      "mean_ber_reduction": "float",
      "mean_gmi_gain": "float",
      "mean_ngmi_gain": "float",
      "mean_deployment_burden": "float",
      "mean_equalizer_benefit": "float",
      "deployment_favorable_fraction": "float"
    }
  },
  "per_row_report": [
    {
      "sample_id": "string",
      "train_split": "string",
      "band": "string",
      "channel_index": "int",
      "channel_in_band": "int",
      "wavelength_nm": "float",
      "ber_reduction": "float",
      "gmi_gain": "float",
      "ngmi_gain": "float",
      "capacity_gain_Gbps": "float",
      "residual_magnitude": "float",
      "uncertainty_score": "float",
      "quality_label": "string",
      "deployment_burden": "float",
      "equalizer_benefit": "float",
      "deployment_favorable": "bool"
    }
  ],
  "bottleneck_analysis": {
    "highest_capacity_gain_sample_id": "string",
    "lowest_capacity_gain_sample_id": "string",
    "lowest_ngmi_sample_id": "string",
    "highest_ber_sample_id": "string",
    "highest_uncertainty_sample_id": "string",
    "highest_deployment_burden_sample_id": "string",
    "best_deployment_sample_id": "string",
    "worst_deployment_sample_id": "string"
  },
  "engineering_conclusion": "string"
}
```
All floating-point values must be JSON numbers, not strings. All row-level arrays and reports must have length equal to the cleaned dataset size and must preserve the cleaned input row order.

This combined version has one clear real-world objective: decide whether the Bayesian CNN-BiLSTM equalizer is useful, how much C+L capacity it enables, where the link is weak, and whether deployment is justified.

## Background

The dataset represents a post-receiver engineering audit for a long-haul coherent optical transmission experiment using a Bayesian CNN-BiLSTM equalizer. The motivating real-world system is a C+L-band transmission setting where neural equalization is used to improve high-capacity long-haul performance; the referenced paper reports **102.3 Tbit/s C+L-band transmission over 1512 km SSMF** enabled by an uncertainty-aware Bayesian CNN-BiLSTM hybrid equalizer.

C+L-band transmission is important because expanding usable optical spectrum is a practical route to higher fiber capacity. Optical-network studies describe C+L or Super C+L transmission as a way to increase network throughput, while also noting physical-layer effects such as stimulated Raman scattering and wavelength-dependent impairments that can make one band or wavelength region weaker than another.

In this task, the no-NLC columns represent the baseline receiver performance without neural nonlinear compensation, and the Bayesian columns represent performance after the Bayesian CNN-BiLSTM equalizer. Positive values of $ \Delta \mathrm{BER}_i $, $ \Delta \mathrm{GMI}_i $, $ \Delta \mathrm{NGMI}_i $, and $ \Delta C_i $ indicate improvement after equalization, except that BER is an error metric, so improvement means the Bayesian BER is lower than the no-NLC BER.

The channel-quality labels are intended for an engineering planning report rather than as universal telecom standards. A `usable` row represents a channel/window whose post-equalization BER and NGMI are both acceptable under the task thresholds. An `unusable` row represents a clearly poor post-equalization observation under the task thresholds. A `marginal` row is an intermediate case that is not clearly usable and not clearly unusable.

The deployment burden is a deterministic relative score within the cleaned dataset, not a measured hardware power value. It should increase with longer equalizer windows, more training epochs, higher symbol rate, stronger memory/nonlinear proxies, stronger power variation, larger uncertainty, or larger target log-variance. Feature scaling may be used for this burden score if it is deterministic and applied to the full cleaned dataset.

The equalizer benefit is also a deterministic relative score within the cleaned dataset. It should increase with larger BER reduction, GMI gain, NGMI gain, capacity gain, and capacity-gain percentage. If all rows have identical values for a feature used in burden or benefit scoring, that feature may contribute a constant or zero normalized term so that outputs remain finite.

The deployment-favorable flag is a practical screening rule: a row must show positive information-rate improvement and have deployment burden no larger than the dataset median burden. This rule is not intended to prove hardware deployability; it identifies observations where equalizer gain appears attractive relative to the estimated computation burden.

The bottleneck identifiers should be selected deterministically from the cleaned row order. If multiple rows tie for a best or worst value, the earliest row in cleaned input order should be used. This tie rule applies to highest capacity gain, lowest capacity gain, lowest NGMI, highest BER, highest uncertainty, highest deployment burden, best deployment sample, and worst deployment sample.

The engineering conclusion should be a concise text summary based only on computed outputs. It should mention whether the Bayesian equalizer improves the link overall, whether C-band or L-band appears stronger in the available rows when both are present, whether the usable-channel count is high or low, and whether deployment appears favorable for a meaningful portion of the dataset.

Neural network equalizers are relevant to coherent optical systems because they can mitigate nonlinear optical-channel impairments, but practical deployment must consider complexity. Literature on neural optical equalizers notes that edge deployment is challenging because efficient nonlinear equalization can require high computational complexity for channels with large dispersion-induced memory, and FPGA-oriented NN equalizer implementation is an active direction.


Deployment burden may be computed as the mean of deterministic full-dataset min-max normalized terms from `window_length`, `epochs_bayesian_cnn_bilstm`, `symbol_rate_GBd`, `isi_memory_proxy`, `nonlinear_phase_proxy`, `window_power_std`, `uncertainty_score`, and `target_log_variance`. If a term is constant across all cleaned rows, its normalized contribution may be zero so that outputs remain finite.

Equalizer benefit may be computed as the mean of deterministic full-dataset min-max normalized nonnegative improvement terms from BER reduction, GMI gain, NGMI gain, capacity gain, and `capacity_gain_pct`.

The best deployment sample may be selected by maximizing `equalizer_benefit - deployment_burden`, and the worst deployment sample may be selected by minimizing the same quantity. This deployment score is only used for ranking the deployment bottleneck identifiers; the row-level output still reports deployment burden, equalizer benefit, and the deployment-favorable flag.

For all deterministic best/worst identifiers, ties are resolved by selecting the earliest row in cleaned input order.

## Solution


In [ ]:
import io
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd
import torch


def solve_cl_band_equalizer_engineering_audit(
    input_path="/workspace/data/cnn_bilstm_equalizer.csv",
    output_path="/workspace/output.json",
):
    required_columns = [
        "sample_id",
        "channel_index",
        "channel_in_band",
        "band",
        "wavelength_nm",
        "distance_km",
        "loop_count",
        "symbol_rate_GBd",
        "modulation",
        "shaping_entropy_bits",
        "window_length",
        "launch_power_dBm",
        "osnr_dB",
        "srs_tilt_dB",
        "nonlinear_phase_proxy",
        "isi_memory_proxy",
        "amplitude_kurtosis_proxy",
        "center_rx_I",
        "center_rx_Q",
        "pre_context_I",
        "pre_context_Q",
        "post_context_I",
        "post_context_Q",
        "window_power_mean",
        "window_power_std",
        "window_iq_corr",
        "lag1_autocorr_I",
        "lag1_autocorr_Q",
        "target_residual_I",
        "target_residual_Q",
        "target_log_variance",
        "uncertainty_score",
        "ber_wo_nlc",
        "ber_bayesian_cnn_bilstm",
        "gmi_wo_nlc_bits_per_symbol",
        "gmi_bayesian_bits_per_symbol",
        "ngmi_wo_nlc",
        "ngmi_bayesian",
        "capacity_wo_nlc_Gbps",
        "capacity_bayesian_Gbps",
        "capacity_gain_pct",
        "epochs_bayesian_cnn_bilstm",
        "train_split",
    ]

    numeric_columns = [
        c
        for c in required_columns
        if c not in {"sample_id", "band", "modulation", "train_split"}
    ]

    tiny = 1e-12

    torch.set_default_dtype(torch.float64)
    torch.set_num_threads(1)
    torch.manual_seed(0)
    torch.use_deterministic_algorithms(True)

    def _read_utf8_text(path):
        text = Path(path).read_text(encoding="utf-8")
        if text.startswith("\ufeff"):
            text = text[1:]
        text = text.replace("\r\n", "\n").replace("\r", "\n")
        return text

    def _load_and_clean_csv(path):
        text = _read_utf8_text(path)
        df = pd.read_csv(io.StringIO(text), dtype=str)

        df.columns = [str(c).strip() for c in df.columns]

        for col in df.columns:
            df[col] = df[col].map(lambda x: x.strip() if isinstance(x, str) else x)

        empty_mask = df.apply(
            lambda col: col.map(lambda x: isinstance(x, str) and x.strip() == "")
        )
        df = df.mask(empty_mask, np.nan)

        df = df.dropna(axis=0, how="all")
        df = df.dropna(axis=1, how="all")

        df.columns = [str(c).strip() for c in df.columns]

        for col in df.columns:
            df[col] = df[col].map(lambda x: x.strip() if isinstance(x, str) else x)

        return df

    def _validate_and_cast(df):
        missing = [c for c in required_columns if c not in df.columns]
        if missing:
            raise ValueError(f"Missing required columns: {missing}")

        df = df[required_columns].copy()

        for col in ["sample_id", "band", "modulation", "train_split"]:
            df[col] = df[col].map(lambda x: str(x).strip() if not pd.isna(x) else x)

        if df["sample_id"].isna().any() or (df["sample_id"] == "").any():
            raise ValueError("sample_id must be non-empty after trimming.")

        if df["sample_id"].duplicated().any():
            raise ValueError("sample_id must be unique after trimming.")

        if df["band"].isna().any() or not set(df["band"].tolist()).issubset({"C", "L"}):
            raise ValueError("band must contain only C or L.")

        if df["modulation"].isna().any() or (df["modulation"] == "").any():
            raise ValueError("modulation must be non-empty.")

        for col in numeric_columns:
            df[col] = pd.to_numeric(df[col], errors="raise")
            values = df[col].to_numpy(dtype=float)
            if not np.all(np.isfinite(values)):
                raise ValueError(f"Column {col} contains non-finite values.")

        positive_columns = [
            "channel_index",
            "channel_in_band",
            "loop_count",
            "symbol_rate_GBd",
            "window_length",
            "epochs_bayesian_cnn_bilstm",
        ]
        for col in positive_columns:
            if np.any(df[col].to_numpy(dtype=float) <= 0):
                raise ValueError(f"Column {col} must be positive.")

        integer_like_columns = [
            "channel_index",
            "channel_in_band",
            "loop_count",
            "window_length",
            "epochs_bayesian_cnn_bilstm",
        ]
        for col in integer_like_columns:
            values = df[col].to_numpy(dtype=float)
            if np.any(np.abs(values - np.round(values)) > 1e-12):
                raise ValueError(f"Column {col} must be integer-valued.")

        nonnegative_columns = [
            "distance_km",
            "shaping_entropy_bits",
            "window_power_mean",
            "window_power_std",
            "uncertainty_score",
            "ber_wo_nlc",
            "ber_bayesian_cnn_bilstm",
            "gmi_wo_nlc_bits_per_symbol",
            "gmi_bayesian_bits_per_symbol",
            "ngmi_wo_nlc",
            "ngmi_bayesian",
            "capacity_wo_nlc_Gbps",
            "capacity_bayesian_Gbps",
        ]
        for col in nonnegative_columns:
            if np.any(df[col].to_numpy(dtype=float) < 0):
                raise ValueError(f"Column {col} must be nonnegative.")

        for col in ["ber_wo_nlc", "ber_bayesian_cnn_bilstm", "ngmi_wo_nlc", "ngmi_bayesian"]:
            values = df[col].to_numpy(dtype=float)
            if np.any(values < 0) or np.any(values > 1):
                raise ValueError(f"Column {col} must lie in [0, 1].")

        for col in ["window_iq_corr", "lag1_autocorr_I", "lag1_autocorr_Q"]:
            values = df[col].to_numpy(dtype=float)
            if np.any(values < -1) or np.any(values > 1):
                raise ValueError(f"Column {col} must lie in [-1, 1].")

        window_values = np.round(df["window_length"].to_numpy(dtype=float)).astype(int)
        if np.any(window_values % 2 == 0):
            raise ValueError("window_length must be odd for every row.")

        if len(df) == 0:
            raise ValueError("The cleaned dataset contains no rows.")

        return df

    def _tensor(values):
        return torch.as_tensor(values, dtype=torch.float64)

    def _minmax(values):
        x = torch.as_tensor(values, dtype=torch.float64)
        lo = torch.min(x)
        hi = torch.max(x)
        denom = torch.clamp(hi - lo, min=tiny)
        return (x - lo) / denom

    def _safe_mean(values):
        values = torch.as_tensor(values, dtype=torch.float64)
        if values.numel() == 0:
            return torch.tensor(0.0, dtype=torch.float64)
        return torch.mean(values)

    def _first_argmax(values):
        arr = np.asarray(values, dtype=float)
        return int(np.argmax(arr))

    def _first_argmin(values):
        arr = np.asarray(values, dtype=float)
        return int(np.argmin(arr))

    def _finite_float(value):
        value = float(value)
        if not math.isfinite(value):
            raise ValueError("Generated non-finite floating-point output.")
        return value

    def _to_builtin(obj):
        if isinstance(obj, dict):
            return {str(k): _to_builtin(v) for k, v in obj.items()}
        if isinstance(obj, list):
            return [_to_builtin(v) for v in obj]
        if isinstance(obj, tuple):
            return [_to_builtin(v) for v in obj]
        if isinstance(obj, np.ndarray):
            return _to_builtin(obj.tolist())
        if isinstance(obj, torch.Tensor):
            return _to_builtin(obj.detach().cpu().numpy())
        if isinstance(obj, (np.integer,)):
            return int(obj)
        if isinstance(obj, (np.floating,)):
            return _finite_float(obj)
        if isinstance(obj, (bool, np.bool_)):
            return bool(obj)
        if isinstance(obj, float):
            return _finite_float(obj)
        return obj

    df = _validate_and_cast(_load_and_clean_csv(input_path))

    sample_ids = df["sample_id"].tolist()
    bands = df["band"].tolist()
    train_splits = df["train_split"].tolist()
    n = len(df)

    data = {
        col: df[col].to_numpy(dtype=float)
        for col in numeric_columns
    }

    ber_wo = _tensor(data["ber_wo_nlc"])
    ber_bayes = _tensor(data["ber_bayesian_cnn_bilstm"])
    gmi_wo = _tensor(data["gmi_wo_nlc_bits_per_symbol"])
    gmi_bayes = _tensor(data["gmi_bayesian_bits_per_symbol"])
    ngmi_wo = _tensor(data["ngmi_wo_nlc"])
    ngmi_bayes = _tensor(data["ngmi_bayesian"])
    cap_wo = _tensor(data["capacity_wo_nlc_Gbps"])
    cap_bayes = _tensor(data["capacity_bayesian_Gbps"])

    ber_reduction = ber_wo - ber_bayes
    gmi_gain = gmi_bayes - gmi_wo
    ngmi_gain = ngmi_bayes - ngmi_wo
    capacity_gain = cap_bayes - cap_wo

    residual_magnitude = torch.sqrt(
        _tensor(data["target_residual_I"]) ** 2
        + _tensor(data["target_residual_Q"]) ** 2
        + tiny
    )

    usable_mask = (ber_bayes <= 0.05) & (ngmi_bayes >= 0.80)
    unusable_mask = (ber_bayes > 0.10) | (ngmi_bayes < 0.70)

    quality_labels = []
    for i in range(n):
        if bool(usable_mask[i].item()):
            quality_labels.append("usable")
        elif bool(unusable_mask[i].item()):
            quality_labels.append("unusable")
        else:
            quality_labels.append("marginal")

    deployment_burden_terms = torch.stack(
        [
            _minmax(data["window_length"]),
            _minmax(data["epochs_bayesian_cnn_bilstm"]),
            _minmax(data["symbol_rate_GBd"]),
            _minmax(data["isi_memory_proxy"]),
            _minmax(data["nonlinear_phase_proxy"]),
            _minmax(data["window_power_std"]),
            _minmax(data["uncertainty_score"]),
            _minmax(data["target_log_variance"]),
        ],
        dim=1,
    )
    deployment_burden = torch.mean(deployment_burden_terms, dim=1)

    equalizer_benefit_terms = torch.stack(
        [
            _minmax(torch.clamp(ber_reduction, min=0.0)),
            _minmax(torch.clamp(gmi_gain, min=0.0)),
            _minmax(torch.clamp(ngmi_gain, min=0.0)),
            _minmax(torch.clamp(capacity_gain, min=0.0)),
            _minmax(torch.clamp(_tensor(data["capacity_gain_pct"]), min=0.0)),
        ],
        dim=1,
    )
    equalizer_benefit = torch.mean(equalizer_benefit_terms, dim=1)

    median_burden = torch.median(deployment_burden)

    deployment_favorable_tensor = (
        (capacity_gain > 0)
        & (gmi_gain > 0)
        & (ngmi_gain > 0)
        & (deployment_burden <= median_burden)
    )

    deployment_score = equalizer_benefit - deployment_burden

    total_bayesian_capacity = torch.sum(cap_bayes)
    total_no_nlc_capacity = torch.sum(cap_wo)
    total_capacity_gain = torch.sum(capacity_gain)

    usable_count = int(torch.sum(usable_mask.to(torch.int64)).item())
    unusable_count = int(torch.sum(unusable_mask.to(torch.int64)).item())
    marginal_count = int(n - usable_count - unusable_count)
    deployment_favorable_count = int(torch.sum(deployment_favorable_tensor.to(torch.int64)).item())

    band_summary = {}
    band_array = np.array(bands)

    for band in ["C", "L"]:
        if not np.any(band_array == band):
            continue

        idx_np = np.where(band_array == band)[0]
        idx = torch.as_tensor(idx_np, dtype=torch.long)

        band_favorable = deployment_favorable_tensor[idx].to(torch.float64)

        band_summary[band] = {
            "row_count": int(len(idx_np)),
            "total_bayesian_capacity_Gbps": _finite_float(torch.sum(cap_bayes[idx]).item()),
            "total_capacity_gain_Gbps": _finite_float(torch.sum(capacity_gain[idx]).item()),
            "mean_ber_reduction": _finite_float(_safe_mean(ber_reduction[idx]).item()),
            "mean_gmi_gain": _finite_float(_safe_mean(gmi_gain[idx]).item()),
            "mean_ngmi_gain": _finite_float(_safe_mean(ngmi_gain[idx]).item()),
            "mean_deployment_burden": _finite_float(_safe_mean(deployment_burden[idx]).item()),
            "mean_equalizer_benefit": _finite_float(_safe_mean(equalizer_benefit[idx]).item()),
            "deployment_favorable_fraction": _finite_float(_safe_mean(band_favorable).item()),
        }

    capacity_gain_np = capacity_gain.detach().cpu().numpy()
    ngmi_bayes_np = ngmi_bayes.detach().cpu().numpy()
    ber_bayes_np = ber_bayes.detach().cpu().numpy()
    uncertainty_np = data["uncertainty_score"]
    deployment_burden_np = deployment_burden.detach().cpu().numpy()
    deployment_score_np = deployment_score.detach().cpu().numpy()

    highest_capacity_gain_idx = _first_argmax(capacity_gain_np)
    lowest_capacity_gain_idx = _first_argmin(capacity_gain_np)
    lowest_ngmi_idx = _first_argmin(ngmi_bayes_np)
    highest_ber_idx = _first_argmax(ber_bayes_np)
    highest_uncertainty_idx = _first_argmax(uncertainty_np)
    highest_deployment_burden_idx = _first_argmax(deployment_burden_np)
    best_deployment_idx = _first_argmax(deployment_score_np)
    worst_deployment_idx = _first_argmin(deployment_score_np)

    per_row_report = []

    ber_reduction_np = ber_reduction.detach().cpu().numpy()
    gmi_gain_np = gmi_gain.detach().cpu().numpy()
    ngmi_gain_np = ngmi_gain.detach().cpu().numpy()
    residual_np = residual_magnitude.detach().cpu().numpy()
    equalizer_benefit_np = equalizer_benefit.detach().cpu().numpy()
    favorable_np = deployment_favorable_tensor.detach().cpu().numpy().astype(bool)

    for i, sample_id in enumerate(sample_ids):
        per_row_report.append(
            {
                "sample_id": str(sample_id),
                "train_split": str(train_splits[i]),
                "band": str(bands[i]),
                "channel_index": int(round(float(data["channel_index"][i]))),
                "channel_in_band": int(round(float(data["channel_in_band"][i]))),
                "wavelength_nm": _finite_float(data["wavelength_nm"][i]),
                "ber_reduction": _finite_float(ber_reduction_np[i]),
                "gmi_gain": _finite_float(gmi_gain_np[i]),
                "ngmi_gain": _finite_float(ngmi_gain_np[i]),
                "capacity_gain_Gbps": _finite_float(capacity_gain_np[i]),
                "residual_magnitude": _finite_float(residual_np[i]),
                "uncertainty_score": _finite_float(uncertainty_np[i]),
                "quality_label": quality_labels[i],
                "deployment_burden": _finite_float(deployment_burden_np[i]),
                "equalizer_benefit": _finite_float(equalizer_benefit_np[i]),
                "deployment_favorable": bool(favorable_np[i]),
            }
        )

    if deployment_favorable_count == 0:
        deployment_phrase = "No row is deployment-favorable under the median-burden screening rule."
    elif deployment_favorable_count == n:
        deployment_phrase = "All rows are deployment-favorable under the median-burden screening rule."
    else:
        deployment_phrase = (
            f"{deployment_favorable_count} of {n} rows are deployment-favorable "
            "under the median-burden screening rule."
        )

    if len(band_summary) == 2:
        c_cap = band_summary["C"]["total_bayesian_capacity_Gbps"]
        l_cap = band_summary["L"]["total_bayesian_capacity_Gbps"]
        if c_cap > l_cap:
            band_phrase = "C-band provides higher Bayesian-equalized capacity in the available rows."
        elif l_cap > c_cap:
            band_phrase = "L-band provides higher Bayesian-equalized capacity in the available rows."
        else:
            band_phrase = "C-band and L-band provide equal Bayesian-equalized capacity in the available rows."
    elif len(band_summary) == 1:
        band_phrase = f"Only {next(iter(band_summary.keys()))}-band rows are present."
    else:
        band_phrase = "No band summary is available."

    if float(total_capacity_gain.item()) > 0:
        gain_phrase = "The Bayesian CNN-BiLSTM equalizer improves total capacity relative to the no-NLC baseline."
    elif float(total_capacity_gain.item()) < 0:
        gain_phrase = "The Bayesian CNN-BiLSTM equalizer reduces total capacity relative to the no-NLC baseline."
    else:
        gain_phrase = "The Bayesian CNN-BiLSTM equalizer has zero net total capacity change relative to the no-NLC baseline."

    engineering_conclusion = (
        f"{gain_phrase} "
        f"Total Bayesian-equalized capacity is {_finite_float(total_bayesian_capacity.item()):.6g} Gbps, "
        f"with {_finite_float(total_capacity_gain.item()):.6g} Gbps total gain over the baseline. "
        f"{usable_count} rows are usable, {marginal_count} are marginal, and {unusable_count} are unusable. "
        f"{band_phrase} "
        f"The weakest bottleneck by NGMI is sample {sample_ids[lowest_ngmi_idx]}, "
        f"while the highest BER is sample {sample_ids[highest_ber_idx]}. "
        f"{deployment_phrase}"
    )

    output = {
        "sample_ids": [str(x) for x in sample_ids],
        "global_summary": {
            "row_count": int(n),
            "total_bayesian_capacity_Gbps": _finite_float(total_bayesian_capacity.item()),
            "total_no_nlc_capacity_Gbps": _finite_float(total_no_nlc_capacity.item()),
            "total_capacity_gain_Gbps": _finite_float(total_capacity_gain.item()),
            "mean_capacity_gain_Gbps": _finite_float(torch.mean(capacity_gain).item()),
            "mean_capacity_gain_pct": _finite_float(torch.mean(_tensor(data["capacity_gain_pct"])).item()),
            "mean_ber_reduction": _finite_float(torch.mean(ber_reduction).item()),
            "mean_gmi_gain": _finite_float(torch.mean(gmi_gain).item()),
            "mean_ngmi_gain": _finite_float(torch.mean(ngmi_gain).item()),
            "usable_channel_count": int(usable_count),
            "marginal_channel_count": int(marginal_count),
            "unusable_channel_count": int(unusable_count),
            "deployment_favorable_count": int(deployment_favorable_count),
            "deployment_favorable_fraction": _finite_float(deployment_favorable_count / float(n)),
        },
        "band_summary": band_summary,
        "per_row_report": per_row_report,
        "bottleneck_analysis": {
            "highest_capacity_gain_sample_id": str(sample_ids[highest_capacity_gain_idx]),
            "lowest_capacity_gain_sample_id": str(sample_ids[lowest_capacity_gain_idx]),
            "lowest_ngmi_sample_id": str(sample_ids[lowest_ngmi_idx]),
            "highest_ber_sample_id": str(sample_ids[highest_ber_idx]),
            "highest_uncertainty_sample_id": str(sample_ids[highest_uncertainty_idx]),
            "highest_deployment_burden_sample_id": str(sample_ids[highest_deployment_burden_idx]),
            "best_deployment_sample_id": str(sample_ids[best_deployment_idx]),
            "worst_deployment_sample_id": str(sample_ids[worst_deployment_idx]),
        },
        "engineering_conclusion": engineering_conclusion,
    }

    output = _to_builtin(output)

    output_file = Path(output_path)
    output_file.parent.mkdir(parents=True, exist_ok=True)
    output_file.write_text(json.dumps(output, indent=2), encoding="utf-8")

    return output

## Testing Template

In [ ]:
from pathlib import Path
import importlib.util
import io
import json
import math
import tempfile

import numpy as np
import pandas as pd
import pytest


DATA_DIR = Path("/workspace")
FUNCTION_FILE = DATA_DIR / "solve_cl_band_equalizer_engineering_audit.py"


REQUIRED_COLUMNS = [
    "sample_id",
    "channel_index",
    "channel_in_band",
    "band",
    "wavelength_nm",
    "distance_km",
    "loop_count",
    "symbol_rate_GBd",
    "modulation",
    "shaping_entropy_bits",
    "window_length",
    "launch_power_dBm",
    "osnr_dB",
    "srs_tilt_dB",
    "nonlinear_phase_proxy",
    "isi_memory_proxy",
    "amplitude_kurtosis_proxy",
    "center_rx_I",
    "center_rx_Q",
    "pre_context_I",
    "pre_context_Q",
    "post_context_I",
    "post_context_Q",
    "window_power_mean",
    "window_power_std",
    "window_iq_corr",
    "lag1_autocorr_I",
    "lag1_autocorr_Q",
    "target_residual_I",
    "target_residual_Q",
    "target_log_variance",
    "uncertainty_score",
    "ber_wo_nlc",
    "ber_bayesian_cnn_bilstm",
    "gmi_wo_nlc_bits_per_symbol",
    "gmi_bayesian_bits_per_symbol",
    "ngmi_wo_nlc",
    "ngmi_bayesian",
    "capacity_wo_nlc_Gbps",
    "capacity_bayesian_Gbps",
    "capacity_gain_pct",
    "epochs_bayesian_cnn_bilstm",
    "train_split",
]


GLOBAL_SUMMARY_KEYS = {
    "row_count",
    "total_bayesian_capacity_Gbps",
    "total_no_nlc_capacity_Gbps",
    "total_capacity_gain_Gbps",
    "mean_capacity_gain_Gbps",
    "mean_capacity_gain_pct",
    "mean_ber_reduction",
    "mean_gmi_gain",
    "mean_ngmi_gain",
    "usable_channel_count",
    "marginal_channel_count",
    "unusable_channel_count",
    "deployment_favorable_count",
    "deployment_favorable_fraction",
}


BAND_SUMMARY_KEYS = {
    "row_count",
    "total_bayesian_capacity_Gbps",
    "total_capacity_gain_Gbps",
    "mean_ber_reduction",
    "mean_gmi_gain",
    "mean_ngmi_gain",
    "mean_deployment_burden",
    "mean_equalizer_benefit",
    "deployment_favorable_fraction",
}


PER_ROW_KEYS = {
    "sample_id",
    "train_split",
    "band",
    "channel_index",
    "channel_in_band",
    "wavelength_nm",
    "ber_reduction",
    "gmi_gain",
    "ngmi_gain",
    "capacity_gain_Gbps",
    "residual_magnitude",
    "uncertainty_score",
    "quality_label",
    "deployment_burden",
    "equalizer_benefit",
    "deployment_favorable",
}


BOTTLENECK_KEYS = {
    "highest_capacity_gain_sample_id",
    "lowest_capacity_gain_sample_id",
    "lowest_ngmi_sample_id",
    "highest_ber_sample_id",
    "highest_uncertainty_sample_id",
    "highest_deployment_burden_sample_id",
    "best_deployment_sample_id",
    "worst_deployment_sample_id",
}


def load_function():
    if not FUNCTION_FILE.exists():
        raise FileNotFoundError(f"{FUNCTION_FILE} does not exist")

    spec = importlib.util.spec_from_file_location(
        "solve_cl_band_equalizer_engineering_audit",
        FUNCTION_FILE,
    )
    if spec is None or spec.loader is None:
        raise ImportError(f"Unable to load module from {FUNCTION_FILE}")

    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module.solve_cl_band_equalizer_engineering_audit


@pytest.fixture(scope="module")
def solve_cl_band_equalizer_engineering_audit():
    return load_function()


def _write_csv(path, text):
    Path(path).write_text(text, encoding="utf-8")


def _run_model(solve_cl_band_equalizer_engineering_audit, csv_text):
    with tempfile.TemporaryDirectory() as tmpdir:
        input_path = Path(tmpdir) / "cnn_bilstm_equalizer.csv"
        output_path = Path(tmpdir) / "output.json"

        _write_csv(input_path, csv_text)

        returned = solve_cl_band_equalizer_engineering_audit(
            input_path=str(input_path),
            output_path=str(output_path),
        )

        if not output_path.exists():
            raise AssertionError("output.json was not written")

        written = json.loads(output_path.read_text(encoding="utf-8"))
        return returned, written


def _run_model_nested_output(solve_cl_band_equalizer_engineering_audit, csv_text):
    with tempfile.TemporaryDirectory() as tmpdir:
        input_path = Path(tmpdir) / "cnn_bilstm_equalizer.csv"
        output_path = Path(tmpdir) / "nested" / "output.json"

        _write_csv(input_path, csv_text)

        returned = solve_cl_band_equalizer_engineering_audit(
            input_path=str(input_path),
            output_path=str(output_path),
        )

        if not output_path.exists():
            raise AssertionError("nested output.json was not written")

        written = json.loads(output_path.read_text(encoding="utf-8"))
        return returned, written


def _assert_invalid_input_rejected(fn):
    try:
        fn()
    except Exception:
        return
    raise AssertionError("Invalid input was not rejected")


def _assert_close(actual, expected, tol=1e-7):
    if abs(float(actual) - float(expected)) > tol:
        raise AssertionError(f"{actual} != {expected} within tol={tol}")


def _is_json_number(x):
    return (
        isinstance(x, (int, float))
        and not isinstance(x, bool)
        and math.isfinite(float(x))
    )


def _all_finite(obj):
    if isinstance(obj, dict):
        return all(_all_finite(v) for v in obj.values())
    if isinstance(obj, list):
        return all(_all_finite(v) for v in obj)
    if isinstance(obj, str) or isinstance(obj, bool) or obj is None:
        return True
    return _is_json_number(obj)


def _parse_clean_reference(csv_text):
    text = csv_text
    if text.startswith("\ufeff"):
        text = text[1:]
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    df = pd.read_csv(io.StringIO(text), dtype=str)
    df.columns = [str(c).strip() for c in df.columns]

    for col in df.columns:
        df[col] = df[col].map(lambda x: x.strip() if isinstance(x, str) else x)

    empty_mask = df.apply(
        lambda col: col.map(lambda x: isinstance(x, str) and x.strip() == "")
    )
    df = df.mask(empty_mask, np.nan)
    df = df.dropna(axis=0, how="all").dropna(axis=1, how="all")

    df.columns = [str(c).strip() for c in df.columns]
    for col in df.columns:
        df[col] = df[col].map(lambda x: x.strip() if isinstance(x, str) else x)

    df = df[REQUIRED_COLUMNS].copy()

    for col in REQUIRED_COLUMNS:
        if col not in {"sample_id", "band", "modulation", "train_split"}:
            df[col] = pd.to_numeric(df[col], errors="raise")

    return df


def _mutate_cell(csv_text, sample_id, column, value):
    df = pd.read_csv(io.StringIO(csv_text), dtype=str)
    if column not in df.columns:
        raise AssertionError(f"Column not found: {column}")
    mask = df["sample_id"].astype(str).str.strip() == sample_id
    if not mask.any():
        raise AssertionError(f"sample_id not found: {sample_id}")
    df.loc[mask, column] = str(value)
    return df.to_csv(index=False)


def _base_valid_csv():
    header = ",".join(REQUIRED_COLUMNS)
    rows = [
        [
            "s1", 1, 1, "C", 1524.50, 1512, 6, 96, "PCS-64-QAM", 5.5, 65,
            22.5, 20.5, -0.4, 0.08, 0.11, 2.90,
            0.12, -0.06, 0.05, -0.02, 0.10, -0.04,
            1.10, 0.14, 0.08, 0.12, 0.10,
            0.020, -0.010, -3.50, 0.18,
            0.030, 0.018, 5.10, 5.37, 0.86, 0.905,
            489.6, 515.5, 5.29, 18, "train",
        ],
        [
            "s2", 2, 2, "C", 1530.10, 1512, 6, 96, "PCS-64-QAM", 5.5, 65,
            22.8, 19.7, -0.6, 0.10, 0.13, 3.00,
            0.14, -0.05, 0.07, -0.01, 0.12, -0.03,
            1.24, 0.19, 0.12, 0.18, 0.15,
            0.024, -0.013, -3.20, 0.22,
            0.036, 0.021, 5.02, 5.31, 0.845, 0.894,
            481.9, 509.8, 5.79, 19, "validation",
        ],
        [
            "s3", 3, 3, "C", 1545.70, 1512, 6, 96, "PCS-64-QAM", 5.5, 65,
            23.0, 18.9, -0.8, 0.13, 0.16, 3.12,
            0.16, -0.03, 0.09, 0.00, 0.15, -0.02,
            1.38, 0.24, 0.16, 0.22, 0.20,
            0.030, -0.017, -2.90, 0.27,
            0.044, 0.026, 4.90, 5.20, 0.825, 0.878,
            470.4, 499.2, 6.12, 21, "test",
        ],
        [
            "s4", 4, 1, "L", 1578.20, 1512, 6, 96, "PCS-64-QAM", 5.5, 65,
            21.4, 18.2, -1.4, 0.17, 0.21, 3.28,
            0.18, 0.02, 0.12, 0.03, 0.17, 0.04,
            1.55, 0.30, 0.21, 0.28, 0.25,
            0.038, -0.020, -2.55, 0.34,
            0.058, 0.034, 4.72, 5.04, 0.798, 0.855,
            453.1, 483.8, 6.78, 22, "train",
        ],
        [
            "s5", 5, 2, "L", 1601.30, 1512, 6, 96, "PCS-64-QAM", 5.5, 65,
            21.1, 17.6, -1.7, 0.20, 0.25, 3.39,
            0.21, 0.04, 0.15, 0.05, 0.20, 0.06,
            1.72, 0.35, 0.27, 0.34, 0.30,
            0.045, -0.026, -2.25, 0.41,
            0.070, 0.041, 4.58, 4.94, 0.774, 0.840,
            439.7, 474.2, 7.85, 23, "validation",
        ],
        [
            "s6", 6, 3, "L", 1620.40, 1512, 6, 96, "PCS-64-QAM", 5.0, 65,
            20.8, 17.0, -1.9, 0.23, 0.28, 3.55,
            0.24, 0.07, 0.17, 0.08, 0.23, 0.10,
            1.91, 0.42, 0.31, 0.40, 0.36,
            0.054, -0.031, -1.95, 0.50,
            0.086, 0.050, 4.38, 4.79, 0.740, 0.815,
            420.5, 459.8, 9.35, 25, "test",
        ],
        [
            "s7", 7, 4, "L", 1630.00, 1512, 6, 96, "PCS-64-QAM", 5.0, 65,
            20.5, 15.2, -2.2, 0.30, 0.35, 3.70,
            0.30, 0.10, 0.20, 0.12, 0.27, 0.15,
            2.05, 0.50, 0.36, 0.46, 0.42,
            0.065, -0.038, -1.60, 0.68,
            0.140, 0.115, 4.05, 4.21, 0.650, 0.680,
            388.8, 404.2, 3.96, 30, "test",
        ],
    ]
    return header + "\n" + "\n".join(",".join(map(str, r)) for r in rows) + "\n"


def _single_band_valid_csv():
    lines = _base_valid_csv().strip().splitlines()
    return "\n".join([lines[0]] + [line for line in lines[1:] if ",C," in line]) + "\n"


def _constant_feature_valid_csv():
    df = pd.read_csv(io.StringIO(_base_valid_csv()), dtype=str)
    constant_cols = [
        "window_length",
        "epochs_bayesian_cnn_bilstm",
        "symbol_rate_GBd",
        "isi_memory_proxy",
        "nonlinear_phase_proxy",
        "window_power_std",
        "uncertainty_score",
        "target_log_variance",
        "capacity_gain_pct",
    ]
    for col in constant_cols:
        df[col] = df[col].iloc[0]
    return df.to_csv(index=False)


def _expected_arrays(csv_text):
    df = _parse_clean_reference(csv_text)

    ber_reduction = df["ber_wo_nlc"].to_numpy(float) - df["ber_bayesian_cnn_bilstm"].to_numpy(float)
    gmi_gain = df["gmi_bayesian_bits_per_symbol"].to_numpy(float) - df["gmi_wo_nlc_bits_per_symbol"].to_numpy(float)
    ngmi_gain = df["ngmi_bayesian"].to_numpy(float) - df["ngmi_wo_nlc"].to_numpy(float)
    capacity_gain = df["capacity_bayesian_Gbps"].to_numpy(float) - df["capacity_wo_nlc_Gbps"].to_numpy(float)
    residual = np.sqrt(
        df["target_residual_I"].to_numpy(float) ** 2
        + df["target_residual_Q"].to_numpy(float) ** 2
    )

    quality = []
    for _, row in df.iterrows():
        ber = float(row["ber_bayesian_cnn_bilstm"])
        ngmi = float(row["ngmi_bayesian"])
        if ber <= 0.05 and ngmi >= 0.80:
            quality.append("usable")
        elif ber > 0.10 or ngmi < 0.70:
            quality.append("unusable")
        else:
            quality.append("marginal")

    return {
        "df": df,
        "sample_ids": df["sample_id"].tolist(),
        "ber_reduction": ber_reduction,
        "gmi_gain": gmi_gain,
        "ngmi_gain": ngmi_gain,
        "capacity_gain": capacity_gain,
        "residual": residual,
        "quality": quality,
    }


def _minmax_np(x):
    x = np.asarray(x, dtype=float)
    lo = float(np.min(x))
    hi = float(np.max(x))
    if abs(hi - lo) <= 1e-12:
        return np.zeros_like(x, dtype=float)
    return (x - lo) / (hi - lo)


def _expected_deployment(csv_text):
    e = _expected_arrays(csv_text)
    df = e["df"]

    burden_terms = np.vstack(
        [
            _minmax_np(df["window_length"].to_numpy(float)),
            _minmax_np(df["epochs_bayesian_cnn_bilstm"].to_numpy(float)),
            _minmax_np(df["symbol_rate_GBd"].to_numpy(float)),
            _minmax_np(df["isi_memory_proxy"].to_numpy(float)),
            _minmax_np(df["nonlinear_phase_proxy"].to_numpy(float)),
            _minmax_np(df["window_power_std"].to_numpy(float)),
            _minmax_np(df["uncertainty_score"].to_numpy(float)),
            _minmax_np(df["target_log_variance"].to_numpy(float)),
        ]
    ).T
    burden = np.mean(burden_terms, axis=1)

    benefit_terms = np.vstack(
        [
            _minmax_np(np.maximum(e["ber_reduction"], 0.0)),
            _minmax_np(np.maximum(e["gmi_gain"], 0.0)),
            _minmax_np(np.maximum(e["ngmi_gain"], 0.0)),
            _minmax_np(np.maximum(e["capacity_gain"], 0.0)),
            _minmax_np(np.maximum(df["capacity_gain_pct"].to_numpy(float), 0.0)),
        ]
    ).T
    benefit = np.mean(benefit_terms, axis=1)

    median_burden = float(np.median(burden))
    favorable = (
        (e["capacity_gain"] > 0.0)
        & (e["gmi_gain"] > 0.0)
        & (e["ngmi_gain"] > 0.0)
        & (burden <= median_burden)
    )
    deployment_score = benefit - burden

    return burden, benefit, favorable, deployment_score


def _assert_schema(out):
    expected_top = {
        "sample_ids",
        "global_summary",
        "band_summary",
        "per_row_report",
        "bottleneck_analysis",
        "engineering_conclusion",
    }
    if set(out.keys()) != expected_top:
        raise AssertionError("Top-level schema mismatch")

    if set(out["global_summary"].keys()) != GLOBAL_SUMMARY_KEYS:
        raise AssertionError("global_summary schema mismatch")

    if set(out["bottleneck_analysis"].keys()) != BOTTLENECK_KEYS:
        raise AssertionError("bottleneck_analysis schema mismatch")

    for row in out["per_row_report"]:
        if set(row.keys()) != PER_ROW_KEYS:
            raise AssertionError("per_row_report schema mismatch")

    for summary in out["band_summary"].values():
        if set(summary.keys()) != BAND_SUMMARY_KEYS:
            raise AssertionError("band_summary schema mismatch")


def test_case_1(solve_cl_band_equalizer_engineering_audit):
    returned, out = _run_model(solve_cl_band_equalizer_engineering_audit, _base_valid_csv())

    if returned != out:
        raise AssertionError("Returned object and written JSON differ")

    _assert_schema(out)

    if not _all_finite(out):
        raise AssertionError("Output contains non-finite values")


def test_case_2(solve_cl_band_equalizer_engineering_audit):
    csv = _base_valid_csv()
    _, out = _run_model(solve_cl_band_equalizer_engineering_audit, csv)
    e = _expected_arrays(csv)

    if out["sample_ids"] != e["sample_ids"]:
        raise AssertionError("sample_ids must preserve cleaned row order")

    if [row["sample_id"] for row in out["per_row_report"]] != e["sample_ids"]:
        raise AssertionError("per_row_report must preserve cleaned row order")


def test_case_3(solve_cl_band_equalizer_engineering_audit):
    header = ",".join([f" {c} " for c in REQUIRED_COLUMNS] + [" empty_extra "])

    row_a_values = [
        " a ", 1, 1, " C ", 1524.5, 1512, 6, 96, " PCS-64-QAM ", 5.5, 65,
        22.5, 20.5, -0.4, 0.08, 0.11, 2.9,
        0.12, -0.06, 0.05, -0.02, 0.10, -0.04,
        1.10, 0.14, 0.08, 0.12, 0.10,
        0.020, -0.010, -3.5, 0.18,
        0.030, 0.018, 5.10, 5.37, 0.86, 0.905,
        489.6, 515.5, 5.29, 18, " train ", ""
    ]

    row_b_values = [
        " b ", 2, 1, " L ", 1578.2, 1512, 6, 96, " PCS-64-QAM ", 5.5, 65,
        21.4, 18.2, -1.4, 0.17, 0.21, 3.28,
        0.18, 0.02, 0.12, 0.03, 0.17, 0.04,
        1.55, 0.30, 0.21, 0.28, 0.25,
        0.038, -0.020, -2.55, 0.34,
        0.058, 0.034, 4.72, 5.04, 0.798, 0.855,
        453.1, 483.8, 6.78, 22, " test ", ""
    ]

    empty_row = [""] * (len(REQUIRED_COLUMNS) + 1)

    csv = (
        "\ufeff" + header + "\r\n"
        + ",".join(map(str, row_a_values)) + "\r\n"
        + ",".join(empty_row) + "\r\n"
        + ",".join(map(str, row_b_values)) + "\n"
    )

    _, out = _run_model(solve_cl_band_equalizer_engineering_audit, csv)

    if out["sample_ids"] != ["a", "b"]:
        raise AssertionError("BOM/whitespace/empty-row/line-ending cleaning failed")

    if [row["train_split"] for row in out["per_row_report"]] != ["train", "test"]:
        raise AssertionError("train_split must be preserved after trimming")


def test_case_4(solve_cl_band_equalizer_engineering_audit):
    returned, written = _run_model_nested_output(
        solve_cl_band_equalizer_engineering_audit,
        _base_valid_csv(),
    )

    if returned != written:
        raise AssertionError("Returned object and nested written JSON differ")

    _assert_schema(written)


def test_case_5(solve_cl_band_equalizer_engineering_audit):
    csv = _base_valid_csv()
    _, out = _run_model(solve_cl_band_equalizer_engineering_audit, csv)
    e = _expected_arrays(csv)

    for i, row in enumerate(out["per_row_report"]):
        _assert_close(row["ber_reduction"], e["ber_reduction"][i])
        _assert_close(row["gmi_gain"], e["gmi_gain"][i])
        _assert_close(row["ngmi_gain"], e["ngmi_gain"][i])
        _assert_close(row["capacity_gain_Gbps"], e["capacity_gain"][i])
        _assert_close(row["residual_magnitude"], e["residual"][i])


def test_case_6(solve_cl_band_equalizer_engineering_audit):
    csv = _base_valid_csv()
    _, out = _run_model(solve_cl_band_equalizer_engineering_audit, csv)
    e = _expected_arrays(csv)

    labels = [row["quality_label"] for row in out["per_row_report"]]
    if labels != e["quality"]:
        raise AssertionError("quality_label values do not match required thresholds")

    if not set(labels).issubset({"usable", "marginal", "unusable"}):
        raise AssertionError("quality_label must be one of usable, marginal, or unusable")


def test_case_7(solve_cl_band_equalizer_engineering_audit):
    csv = _base_valid_csv()
    _, out = _run_model(solve_cl_band_equalizer_engineering_audit, csv)
    e = _expected_arrays(csv)

    gs = out["global_summary"]
    _assert_close(gs["total_bayesian_capacity_Gbps"], e["df"]["capacity_bayesian_Gbps"].sum())
    _assert_close(gs["total_no_nlc_capacity_Gbps"], e["df"]["capacity_wo_nlc_Gbps"].sum())
    _assert_close(gs["total_capacity_gain_Gbps"], np.sum(e["capacity_gain"]))
    _assert_close(gs["mean_capacity_gain_Gbps"], np.mean(e["capacity_gain"]))
    _assert_close(gs["mean_capacity_gain_pct"], e["df"]["capacity_gain_pct"].mean())
    _assert_close(gs["mean_ber_reduction"], np.mean(e["ber_reduction"]))
    _assert_close(gs["mean_gmi_gain"], np.mean(e["gmi_gain"]))
    _assert_close(gs["mean_ngmi_gain"], np.mean(e["ngmi_gain"]))


def test_case_8(solve_cl_band_equalizer_engineering_audit):
    csv = _base_valid_csv()
    _, out = _run_model(solve_cl_band_equalizer_engineering_audit, csv)
    e = _expected_arrays(csv)

    gs = out["global_summary"]
    assert gs["row_count"] == len(e["df"])
    assert gs["usable_channel_count"] == e["quality"].count("usable")
    assert gs["marginal_channel_count"] == e["quality"].count("marginal")
    assert gs["unusable_channel_count"] == e["quality"].count("unusable")
    assert (
        gs["usable_channel_count"]
        + gs["marginal_channel_count"]
        + gs["unusable_channel_count"]
        == gs["row_count"]
    )


def test_case_9(solve_cl_band_equalizer_engineering_audit):
    csv = _base_valid_csv()
    _, out = _run_model(solve_cl_band_equalizer_engineering_audit, csv)
    e = _expected_arrays(csv)
    burden, benefit, favorable, _ = _expected_deployment(csv)

    rows = out["per_row_report"]
    for i, row in enumerate(rows):
        _assert_close(row["deployment_burden"], burden[i])
        _assert_close(row["equalizer_benefit"], benefit[i])
        if row["deployment_favorable"] != bool(favorable[i]):
            raise AssertionError("deployment_favorable does not match required rule")

    assert out["global_summary"]["deployment_favorable_count"] == int(np.sum(favorable))
    _assert_close(
        out["global_summary"]["deployment_favorable_fraction"],
        float(np.mean(favorable)),
    )


def test_case_10(solve_cl_band_equalizer_engineering_audit):
    csv = _constant_feature_valid_csv()
    _, out = _run_model(solve_cl_band_equalizer_engineering_audit, csv)

    for row in out["per_row_report"]:
        if row["deployment_burden"] < -1e-12:
            raise AssertionError("deployment_burden must be nonnegative")
        if row["equalizer_benefit"] < -1e-12:
            raise AssertionError("equalizer_benefit must be nonnegative")
        if not math.isfinite(row["deployment_burden"]):
            raise AssertionError("deployment_burden must remain finite for constant features")
        if not math.isfinite(row["equalizer_benefit"]):
            raise AssertionError("equalizer_benefit must remain finite for constant features")


def test_case_11(solve_cl_band_equalizer_engineering_audit):
    csv = _base_valid_csv()
    _, out = _run_model(solve_cl_band_equalizer_engineering_audit, csv)
    e = _expected_arrays(csv)
    burden, benefit, favorable, _ = _expected_deployment(csv)

    expected_bands = set(e["df"]["band"])
    if set(out["band_summary"].keys()) != expected_bands:
        raise AssertionError("band_summary must contain all and only present bands")

    for band in expected_bands:
        mask = e["df"]["band"].to_numpy() == band
        summary = out["band_summary"][band]
        assert summary["row_count"] == int(np.sum(mask))
        _assert_close(summary["total_bayesian_capacity_Gbps"], e["df"]["capacity_bayesian_Gbps"].to_numpy(float)[mask].sum())
        _assert_close(summary["total_capacity_gain_Gbps"], e["capacity_gain"][mask].sum())
        _assert_close(summary["mean_ber_reduction"], e["ber_reduction"][mask].mean())
        _assert_close(summary["mean_gmi_gain"], e["gmi_gain"][mask].mean())
        _assert_close(summary["mean_ngmi_gain"], e["ngmi_gain"][mask].mean())
        _assert_close(summary["mean_deployment_burden"], burden[mask].mean())
        _assert_close(summary["mean_equalizer_benefit"], benefit[mask].mean())
        _assert_close(summary["deployment_favorable_fraction"], favorable[mask].mean())


def test_case_12(solve_cl_band_equalizer_engineering_audit):
    csv = _single_band_valid_csv()
    _, out = _run_model(solve_cl_band_equalizer_engineering_audit, csv)

    if set(out["band_summary"].keys()) != {"C"}:
        raise AssertionError("Single-band input should summarize only the present band")

    if len(out["per_row_report"]) != 3:
        raise AssertionError("Single-band input should still use all cleaned rows")


def test_case_13(solve_cl_band_equalizer_engineering_audit):
    csv = _base_valid_csv()
    _, out = _run_model(solve_cl_band_equalizer_engineering_audit, csv)
    e = _expected_arrays(csv)
    burden, benefit, favorable, deployment_score = _expected_deployment(csv)

    ba = out["bottleneck_analysis"]

    assert ba["highest_capacity_gain_sample_id"] == e["sample_ids"][int(np.argmax(e["capacity_gain"]))]
    assert ba["lowest_capacity_gain_sample_id"] == e["sample_ids"][int(np.argmin(e["capacity_gain"]))]
    assert ba["lowest_ngmi_sample_id"] == e["sample_ids"][int(np.argmin(e["df"]["ngmi_bayesian"].to_numpy(float)))]
    assert ba["highest_ber_sample_id"] == e["sample_ids"][int(np.argmax(e["df"]["ber_bayesian_cnn_bilstm"].to_numpy(float)))]
    assert ba["highest_uncertainty_sample_id"] == e["sample_ids"][int(np.argmax(e["df"]["uncertainty_score"].to_numpy(float)))]
    assert ba["highest_deployment_burden_sample_id"] == e["sample_ids"][int(np.argmax(burden))]
    assert ba["best_deployment_sample_id"] == e["sample_ids"][int(np.argmax(deployment_score))]
    assert ba["worst_deployment_sample_id"] == e["sample_ids"][int(np.argmin(deployment_score))]


def test_case_14(solve_cl_band_equalizer_engineering_audit):
    csv = _base_valid_csv()
    csv = _mutate_cell(csv, "s1", "capacity_bayesian_Gbps", 600.0)
    csv = _mutate_cell(csv, "s2", "capacity_bayesian_Gbps", 600.0)
    csv = _mutate_cell(csv, "s1", "capacity_wo_nlc_Gbps", 500.0)
    csv = _mutate_cell(csv, "s2", "capacity_wo_nlc_Gbps", 500.0)

    _, out = _run_model(solve_cl_band_equalizer_engineering_audit, csv)

    if out["bottleneck_analysis"]["highest_capacity_gain_sample_id"] != "s1":
        raise AssertionError("Ties for best/worst identifiers must use earliest cleaned row order")


def test_case_15(solve_cl_band_equalizer_engineering_audit):
    csv = _base_valid_csv()
    _, out = _run_model(solve_cl_band_equalizer_engineering_audit, csv)

    conclusion = out["engineering_conclusion"]
    if not isinstance(conclusion, str) or not conclusion.strip():
        raise AssertionError("engineering_conclusion must be non-empty text")

    required_mentions = [
        str(out["global_summary"]["usable_channel_count"]),
        out["bottleneck_analysis"]["lowest_ngmi_sample_id"],
        out["bottleneck_analysis"]["highest_ber_sample_id"],
    ]
    for text in required_mentions:
        if text not in conclusion:
            raise AssertionError("engineering_conclusion should be based on computed outputs")


def test_case_16(solve_cl_band_equalizer_engineering_audit):
    csv = _base_valid_csv()
    _, out1 = _run_model(solve_cl_band_equalizer_engineering_audit, csv)
    _, out2 = _run_model(solve_cl_band_equalizer_engineering_audit, csv)

    if out1 != out2:
        raise AssertionError("Solution must be deterministic for the same input")


def test_case_17(solve_cl_band_equalizer_engineering_audit):
    csv_base = _base_valid_csv()
    csv_changed = _mutate_cell(csv_base, "s1", "capacity_bayesian_Gbps", 600.0)

    _, out_base = _run_model(solve_cl_band_equalizer_engineering_audit, csv_base)
    _, out_changed = _run_model(solve_cl_band_equalizer_engineering_audit, csv_changed)

    if out_base["global_summary"]["total_bayesian_capacity_Gbps"] == out_changed["global_summary"]["total_bayesian_capacity_Gbps"]:
        raise AssertionError("Capacity totals must respond to capacity_bayesian_Gbps changes")


def test_case_18(solve_cl_band_equalizer_engineering_audit):
    csv_base = _base_valid_csv()
    csv_changed = _mutate_cell(csv_base, "s1", "ber_bayesian_cnn_bilstm", 0.004)

    _, out_base = _run_model(solve_cl_band_equalizer_engineering_audit, csv_base)
    _, out_changed = _run_model(solve_cl_band_equalizer_engineering_audit, csv_changed)

    if out_base["per_row_report"][0]["ber_reduction"] == out_changed["per_row_report"][0]["ber_reduction"]:
        raise AssertionError("BER reduction must respond to Bayesian BER changes")


def test_case_19(solve_cl_band_equalizer_engineering_audit):
    csv_base = _base_valid_csv()
    csv_changed = _mutate_cell(csv_base, "s1", "window_length", 129)

    _, out_base = _run_model(solve_cl_band_equalizer_engineering_audit, csv_base)
    _, out_changed = _run_model(solve_cl_band_equalizer_engineering_audit, csv_changed)

    burdens_base = [row["deployment_burden"] for row in out_base["per_row_report"]]
    burdens_changed = [row["deployment_burden"] for row in out_changed["per_row_report"]]

    if burdens_base == burdens_changed:
        raise AssertionError("Deployment burden must respond to required burden features")


def test_case_20(solve_cl_band_equalizer_engineering_audit):
    csv_base = _base_valid_csv()
    csv_changed = _mutate_cell(csv_base, "s1", "capacity_gain_pct", 20.0)

    _, out_base = _run_model(solve_cl_band_equalizer_engineering_audit, csv_base)
    _, out_changed = _run_model(solve_cl_band_equalizer_engineering_audit, csv_changed)

    benefits_base = [row["equalizer_benefit"] for row in out_base["per_row_report"]]
    benefits_changed = [row["equalizer_benefit"] for row in out_changed["per_row_report"]]

    if benefits_base == benefits_changed:
        raise AssertionError("Equalizer benefit must respond to required benefit features")


def test_case_21(solve_cl_band_equalizer_engineering_audit):
    csv = ",".join(REQUIRED_COLUMNS[:8]) + "\n"
    csv += "s1,1,1,C,1524.5,1512,6,96\n"
    _assert_invalid_input_rejected(lambda: _run_model(solve_cl_band_equalizer_engineering_audit, csv))


def test_case_22(solve_cl_band_equalizer_engineering_audit):
    csv = _mutate_cell(_base_valid_csv(), "s2", "sample_id", "s1")
    _assert_invalid_input_rejected(lambda: _run_model(solve_cl_band_equalizer_engineering_audit, csv))


def test_case_23(solve_cl_band_equalizer_engineering_audit):
    csv = _mutate_cell(_base_valid_csv(), "s2", "sample_id", "   ")
    _assert_invalid_input_rejected(lambda: _run_model(solve_cl_band_equalizer_engineering_audit, csv))


def test_case_24(solve_cl_band_equalizer_engineering_audit):
    csv = _mutate_cell(_base_valid_csv(), "s1", "band", "S")
    _assert_invalid_input_rejected(lambda: _run_model(solve_cl_band_equalizer_engineering_audit, csv))


def test_case_25(solve_cl_band_equalizer_engineering_audit):
    csv = _mutate_cell(_base_valid_csv(), "s1", "modulation", "   ")
    _assert_invalid_input_rejected(lambda: _run_model(solve_cl_band_equalizer_engineering_audit, csv))


def test_case_26(solve_cl_band_equalizer_engineering_audit):
    csv = _mutate_cell(_base_valid_csv(), "s1", "channel_index", "not_a_number")
    _assert_invalid_input_rejected(lambda: _run_model(solve_cl_band_equalizer_engineering_audit, csv))


def test_case_27(solve_cl_band_equalizer_engineering_audit):
    csv = _mutate_cell(_base_valid_csv(), "s1", "channel_index", "inf")
    _assert_invalid_input_rejected(lambda: _run_model(solve_cl_band_equalizer_engineering_audit, csv))


def test_case_28(solve_cl_band_equalizer_engineering_audit):
    csv = _mutate_cell(_base_valid_csv(), "s1", "channel_index", 0)
    _assert_invalid_input_rejected(lambda: _run_model(solve_cl_band_equalizer_engineering_audit, csv))


def test_case_29(solve_cl_band_equalizer_engineering_audit):
    csv = _mutate_cell(_base_valid_csv(), "s1", "distance_km", -1)
    _assert_invalid_input_rejected(lambda: _run_model(solve_cl_band_equalizer_engineering_audit, csv))


def test_case_30(solve_cl_band_equalizer_engineering_audit):
    csv = _mutate_cell(_base_valid_csv(), "s1", "ber_bayesian_cnn_bilstm", 1.3)
    _assert_invalid_input_rejected(lambda: _run_model(solve_cl_band_equalizer_engineering_audit, csv))


def test_case_31(solve_cl_band_equalizer_engineering_audit):
    csv = _mutate_cell(_base_valid_csv(), "s1", "ngmi_bayesian", -0.1)
    _assert_invalid_input_rejected(lambda: _run_model(solve_cl_band_equalizer_engineering_audit, csv))


def test_case_32(solve_cl_band_equalizer_engineering_audit):
    csv = _mutate_cell(_base_valid_csv(), "s1", "window_iq_corr", 1.2)
    _assert_invalid_input_rejected(lambda: _run_model(solve_cl_band_equalizer_engineering_audit, csv))


def test_case_33(solve_cl_band_equalizer_engineering_audit):
    csv = _mutate_cell(_base_valid_csv(), "s1", "lag1_autocorr_I", -1.2)
    _assert_invalid_input_rejected(lambda: _run_model(solve_cl_band_equalizer_engineering_audit, csv))


def test_case_34(solve_cl_band_equalizer_engineering_audit):
    csv = _mutate_cell(_base_valid_csv(), "s1", "window_length", 64)
    _assert_invalid_input_rejected(lambda: _run_model(solve_cl_band_equalizer_engineering_audit, csv))


def test_case_35(solve_cl_band_equalizer_engineering_audit):
    csv = _mutate_cell(_base_valid_csv(), "s1", "window_length", 65.5)
    _assert_invalid_input_rejected(lambda: _run_model(solve_cl_band_equalizer_engineering_audit, csv))


def test_case_36(solve_cl_band_equalizer_engineering_audit):
    csv = _mutate_cell(_base_valid_csv(), "s1", "epochs_bayesian_cnn_bilstm", 0)
    _assert_invalid_input_rejected(lambda: _run_model(solve_cl_band_equalizer_engineering_audit, csv))


def test_case_37(solve_cl_band_equalizer_engineering_audit):
    csv = _mutate_cell(_base_valid_csv(), "s1", "capacity_bayesian_Gbps", -2)
    _assert_invalid_input_rejected(lambda: _run_model(solve_cl_band_equalizer_engineering_audit, csv))


def test_case_38(solve_cl_band_equalizer_engineering_audit):
    csv = (
        ",".join(REQUIRED_COLUMNS + ["unused_empty"]) + "\n"
        "x1,1,1,C,1524.5,1512,6,96,PCS-64-QAM,5.5,65,22.5,20.5,-0.4,0.08,0.11,2.90,0.12,-0.06,0.05,-0.02,0.10,-0.04,1.10,0.14,0.08,0.12,0.10,0.020,-0.010,-3.5,0.18,0.030,0.018,5.10,5.37,0.86,0.905,489.6,515.5,5.29,18,train,\n"
        "x2,2,1,L,1578.2,1512,6,96,PCS-64-QAM,5.5,65,21.4,18.2,-1.4,0.17,0.21,3.28,0.18,0.02,0.12,0.03,0.17,0.04,1.55,0.30,0.21,0.28,0.25,0.038,-0.020,-2.55,0.34,0.058,0.034,4.72,5.04,0.798,0.855,453.1,483.8,6.78,22,test,\n"
    )
    _, out = _run_model(solve_cl_band_equalizer_engineering_audit, csv)

    if out["sample_ids"] != ["x1", "x2"]:
        raise AssertionError("Fully empty additional columns should be ignored after cleaning")

In [ ]:
solution_code = r"""
import io
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd
import torch


def solve_cl_band_equalizer_engineering_audit(
    input_path="/workspace/data/cnn_bilstm_equalizer.csv",
    output_path="/workspace/output.json",
):
    required_columns = [
        "sample_id",
        "channel_index",
        "channel_in_band",
        "band",
        "wavelength_nm",
        "distance_km",
        "loop_count",
        "symbol_rate_GBd",
        "modulation",
        "shaping_entropy_bits",
        "window_length",
        "launch_power_dBm",
        "osnr_dB",
        "srs_tilt_dB",
        "nonlinear_phase_proxy",
        "isi_memory_proxy",
        "amplitude_kurtosis_proxy",
        "center_rx_I",
        "center_rx_Q",
        "pre_context_I",
        "pre_context_Q",
        "post_context_I",
        "post_context_Q",
        "window_power_mean",
        "window_power_std",
        "window_iq_corr",
        "lag1_autocorr_I",
        "lag1_autocorr_Q",
        "target_residual_I",
        "target_residual_Q",
        "target_log_variance",
        "uncertainty_score",
        "ber_wo_nlc",
        "ber_bayesian_cnn_bilstm",
        "gmi_wo_nlc_bits_per_symbol",
        "gmi_bayesian_bits_per_symbol",
        "ngmi_wo_nlc",
        "ngmi_bayesian",
        "capacity_wo_nlc_Gbps",
        "capacity_bayesian_Gbps",
        "capacity_gain_pct",
        "epochs_bayesian_cnn_bilstm",
        "train_split",
    ]

    numeric_columns = [
        c
        for c in required_columns
        if c not in {"sample_id", "band", "modulation", "train_split"}
    ]

    tiny = 1e-12

    torch.set_default_dtype(torch.float64)
    torch.set_num_threads(1)
    torch.manual_seed(0)
    torch.use_deterministic_algorithms(True)

    def _read_utf8_text(path):
        text = Path(path).read_text(encoding="utf-8")
        if text.startswith("\ufeff"):
            text = text[1:]
        text = text.replace("\r\n", "\n").replace("\r", "\n")
        return text

    def _load_and_clean_csv(path):
        text = _read_utf8_text(path)
        df = pd.read_csv(io.StringIO(text), dtype=str)

        df.columns = [str(c).strip() for c in df.columns]

        for col in df.columns:
            df[col] = df[col].map(lambda x: x.strip() if isinstance(x, str) else x)

        empty_mask = df.apply(
            lambda col: col.map(lambda x: isinstance(x, str) and x.strip() == "")
        )
        df = df.mask(empty_mask, np.nan)

        df = df.dropna(axis=0, how="all")
        df = df.dropna(axis=1, how="all")

        df.columns = [str(c).strip() for c in df.columns]

        for col in df.columns:
            df[col] = df[col].map(lambda x: x.strip() if isinstance(x, str) else x)

        return df

    def _validate_and_cast(df):
        missing = [c for c in required_columns if c not in df.columns]
        if missing:
            raise ValueError(f"Missing required columns: {missing}")

        df = df[required_columns].copy()

        for col in ["sample_id", "band", "modulation", "train_split"]:
            df[col] = df[col].map(lambda x: str(x).strip() if not pd.isna(x) else x)

        if df["sample_id"].isna().any() or (df["sample_id"] == "").any():
            raise ValueError("sample_id must be non-empty after trimming.")

        if df["sample_id"].duplicated().any():
            raise ValueError("sample_id must be unique after trimming.")

        if df["band"].isna().any() or not set(df["band"].tolist()).issubset({"C", "L"}):
            raise ValueError("band must contain only C or L.")

        if df["modulation"].isna().any() or (df["modulation"] == "").any():
            raise ValueError("modulation must be non-empty.")

        for col in numeric_columns:
            df[col] = pd.to_numeric(df[col], errors="raise")
            values = df[col].to_numpy(dtype=float)
            if not np.all(np.isfinite(values)):
                raise ValueError(f"Column {col} contains non-finite values.")

        positive_columns = [
            "channel_index",
            "channel_in_band",
            "loop_count",
            "symbol_rate_GBd",
            "window_length",
            "epochs_bayesian_cnn_bilstm",
        ]
        for col in positive_columns:
            if np.any(df[col].to_numpy(dtype=float) <= 0):
                raise ValueError(f"Column {col} must be positive.")

        integer_like_columns = [
            "channel_index",
            "channel_in_band",
            "loop_count",
            "window_length",
            "epochs_bayesian_cnn_bilstm",
        ]
        for col in integer_like_columns:
            values = df[col].to_numpy(dtype=float)
            if np.any(np.abs(values - np.round(values)) > 1e-12):
                raise ValueError(f"Column {col} must be integer-valued.")

        nonnegative_columns = [
            "distance_km",
            "shaping_entropy_bits",
            "window_power_mean",
            "window_power_std",
            "uncertainty_score",
            "ber_wo_nlc",
            "ber_bayesian_cnn_bilstm",
            "gmi_wo_nlc_bits_per_symbol",
            "gmi_bayesian_bits_per_symbol",
            "ngmi_wo_nlc",
            "ngmi_bayesian",
            "capacity_wo_nlc_Gbps",
            "capacity_bayesian_Gbps",
        ]
        for col in nonnegative_columns:
            if np.any(df[col].to_numpy(dtype=float) < 0):
                raise ValueError(f"Column {col} must be nonnegative.")

        for col in ["ber_wo_nlc", "ber_bayesian_cnn_bilstm", "ngmi_wo_nlc", "ngmi_bayesian"]:
            values = df[col].to_numpy(dtype=float)
            if np.any(values < 0) or np.any(values > 1):
                raise ValueError(f"Column {col} must lie in [0, 1].")

        for col in ["window_iq_corr", "lag1_autocorr_I", "lag1_autocorr_Q"]:
            values = df[col].to_numpy(dtype=float)
            if np.any(values < -1) or np.any(values > 1):
                raise ValueError(f"Column {col} must lie in [-1, 1].")

        window_values = np.round(df["window_length"].to_numpy(dtype=float)).astype(int)
        if np.any(window_values % 2 == 0):
            raise ValueError("window_length must be odd for every row.")

        if len(df) == 0:
            raise ValueError("The cleaned dataset contains no rows.")

        return df

    def _tensor(values):
        return torch.as_tensor(values, dtype=torch.float64)

    def _minmax(values):
        x = torch.as_tensor(values, dtype=torch.float64)
        lo = torch.min(x)
        hi = torch.max(x)
        denom = torch.clamp(hi - lo, min=tiny)
        return (x - lo) / denom

    def _safe_mean(values):
        values = torch.as_tensor(values, dtype=torch.float64)
        if values.numel() == 0:
            return torch.tensor(0.0, dtype=torch.float64)
        return torch.mean(values)

    def _first_argmax(values):
        arr = np.asarray(values, dtype=float)
        return int(np.argmax(arr))

    def _first_argmin(values):
        arr = np.asarray(values, dtype=float)
        return int(np.argmin(arr))

    def _finite_float(value):
        value = float(value)
        if not math.isfinite(value):
            raise ValueError("Generated non-finite floating-point output.")
        return value

    def _to_builtin(obj):
        if isinstance(obj, dict):
            return {str(k): _to_builtin(v) for k, v in obj.items()}
        if isinstance(obj, list):
            return [_to_builtin(v) for v in obj]
        if isinstance(obj, tuple):
            return [_to_builtin(v) for v in obj]
        if isinstance(obj, np.ndarray):
            return _to_builtin(obj.tolist())
        if isinstance(obj, torch.Tensor):
            return _to_builtin(obj.detach().cpu().numpy())
        if isinstance(obj, (np.integer,)):
            return int(obj)
        if isinstance(obj, (np.floating,)):
            return _finite_float(obj)
        if isinstance(obj, (bool, np.bool_)):
            return bool(obj)
        if isinstance(obj, float):
            return _finite_float(obj)
        return obj

    df = _validate_and_cast(_load_and_clean_csv(input_path))

    sample_ids = df["sample_id"].tolist()
    bands = df["band"].tolist()
    train_splits = df["train_split"].tolist()
    n = len(df)

    data = {
        col: df[col].to_numpy(dtype=float)
        for col in numeric_columns
    }

    ber_wo = _tensor(data["ber_wo_nlc"])
    ber_bayes = _tensor(data["ber_bayesian_cnn_bilstm"])
    gmi_wo = _tensor(data["gmi_wo_nlc_bits_per_symbol"])
    gmi_bayes = _tensor(data["gmi_bayesian_bits_per_symbol"])
    ngmi_wo = _tensor(data["ngmi_wo_nlc"])
    ngmi_bayes = _tensor(data["ngmi_bayesian"])
    cap_wo = _tensor(data["capacity_wo_nlc_Gbps"])
    cap_bayes = _tensor(data["capacity_bayesian_Gbps"])

    ber_reduction = ber_wo - ber_bayes
    gmi_gain = gmi_bayes - gmi_wo
    ngmi_gain = ngmi_bayes - ngmi_wo
    capacity_gain = cap_bayes - cap_wo

    residual_magnitude = torch.sqrt(
        _tensor(data["target_residual_I"]) ** 2
        + _tensor(data["target_residual_Q"]) ** 2
        + tiny
    )

    usable_mask = (ber_bayes <= 0.05) & (ngmi_bayes >= 0.80)
    unusable_mask = (ber_bayes > 0.10) | (ngmi_bayes < 0.70)

    quality_labels = []
    for i in range(n):
        if bool(usable_mask[i].item()):
            quality_labels.append("usable")
        elif bool(unusable_mask[i].item()):
            quality_labels.append("unusable")
        else:
            quality_labels.append("marginal")

    deployment_burden_terms = torch.stack(
        [
            _minmax(data["window_length"]),
            _minmax(data["epochs_bayesian_cnn_bilstm"]),
            _minmax(data["symbol_rate_GBd"]),
            _minmax(data["isi_memory_proxy"]),
            _minmax(data["nonlinear_phase_proxy"]),
            _minmax(data["window_power_std"]),
            _minmax(data["uncertainty_score"]),
            _minmax(data["target_log_variance"]),
        ],
        dim=1,
    )
    deployment_burden = torch.mean(deployment_burden_terms, dim=1)

    equalizer_benefit_terms = torch.stack(
        [
            _minmax(torch.clamp(ber_reduction, min=0.0)),
            _minmax(torch.clamp(gmi_gain, min=0.0)),
            _minmax(torch.clamp(ngmi_gain, min=0.0)),
            _minmax(torch.clamp(capacity_gain, min=0.0)),
            _minmax(torch.clamp(_tensor(data["capacity_gain_pct"]), min=0.0)),
        ],
        dim=1,
    )
    equalizer_benefit = torch.mean(equalizer_benefit_terms, dim=1)

    median_burden = torch.median(deployment_burden)

    deployment_favorable_tensor = (
        (capacity_gain > 0)
        & (gmi_gain > 0)
        & (ngmi_gain > 0)
        & (deployment_burden <= median_burden)
    )

    deployment_score = equalizer_benefit - deployment_burden

    total_bayesian_capacity = torch.sum(cap_bayes)
    total_no_nlc_capacity = torch.sum(cap_wo)
    total_capacity_gain = torch.sum(capacity_gain)

    usable_count = int(torch.sum(usable_mask.to(torch.int64)).item())
    unusable_count = int(torch.sum(unusable_mask.to(torch.int64)).item())
    marginal_count = int(n - usable_count - unusable_count)
    deployment_favorable_count = int(torch.sum(deployment_favorable_tensor.to(torch.int64)).item())

    band_summary = {}
    band_array = np.array(bands)

    for band in ["C", "L"]:
        if not np.any(band_array == band):
            continue

        idx_np = np.where(band_array == band)[0]
        idx = torch.as_tensor(idx_np, dtype=torch.long)

        band_favorable = deployment_favorable_tensor[idx].to(torch.float64)

        band_summary[band] = {
            "row_count": int(len(idx_np)),
            "total_bayesian_capacity_Gbps": _finite_float(torch.sum(cap_bayes[idx]).item()),
            "total_capacity_gain_Gbps": _finite_float(torch.sum(capacity_gain[idx]).item()),
            "mean_ber_reduction": _finite_float(_safe_mean(ber_reduction[idx]).item()),
            "mean_gmi_gain": _finite_float(_safe_mean(gmi_gain[idx]).item()),
            "mean_ngmi_gain": _finite_float(_safe_mean(ngmi_gain[idx]).item()),
            "mean_deployment_burden": _finite_float(_safe_mean(deployment_burden[idx]).item()),
            "mean_equalizer_benefit": _finite_float(_safe_mean(equalizer_benefit[idx]).item()),
            "deployment_favorable_fraction": _finite_float(_safe_mean(band_favorable).item()),
        }

    capacity_gain_np = capacity_gain.detach().cpu().numpy()
    ngmi_bayes_np = ngmi_bayes.detach().cpu().numpy()
    ber_bayes_np = ber_bayes.detach().cpu().numpy()
    uncertainty_np = data["uncertainty_score"]
    deployment_burden_np = deployment_burden.detach().cpu().numpy()
    deployment_score_np = deployment_score.detach().cpu().numpy()

    highest_capacity_gain_idx = _first_argmax(capacity_gain_np)
    lowest_capacity_gain_idx = _first_argmin(capacity_gain_np)
    lowest_ngmi_idx = _first_argmin(ngmi_bayes_np)
    highest_ber_idx = _first_argmax(ber_bayes_np)
    highest_uncertainty_idx = _first_argmax(uncertainty_np)
    highest_deployment_burden_idx = _first_argmax(deployment_burden_np)
    best_deployment_idx = _first_argmax(deployment_score_np)
    worst_deployment_idx = _first_argmin(deployment_score_np)

    per_row_report = []

    ber_reduction_np = ber_reduction.detach().cpu().numpy()
    gmi_gain_np = gmi_gain.detach().cpu().numpy()
    ngmi_gain_np = ngmi_gain.detach().cpu().numpy()
    residual_np = residual_magnitude.detach().cpu().numpy()
    equalizer_benefit_np = equalizer_benefit.detach().cpu().numpy()
    favorable_np = deployment_favorable_tensor.detach().cpu().numpy().astype(bool)

    for i, sample_id in enumerate(sample_ids):
        per_row_report.append(
            {
                "sample_id": str(sample_id),
                "train_split": str(train_splits[i]),
                "band": str(bands[i]),
                "channel_index": int(round(float(data["channel_index"][i]))),
                "channel_in_band": int(round(float(data["channel_in_band"][i]))),
                "wavelength_nm": _finite_float(data["wavelength_nm"][i]),
                "ber_reduction": _finite_float(ber_reduction_np[i]),
                "gmi_gain": _finite_float(gmi_gain_np[i]),
                "ngmi_gain": _finite_float(ngmi_gain_np[i]),
                "capacity_gain_Gbps": _finite_float(capacity_gain_np[i]),
                "residual_magnitude": _finite_float(residual_np[i]),
                "uncertainty_score": _finite_float(uncertainty_np[i]),
                "quality_label": quality_labels[i],
                "deployment_burden": _finite_float(deployment_burden_np[i]),
                "equalizer_benefit": _finite_float(equalizer_benefit_np[i]),
                "deployment_favorable": bool(favorable_np[i]),
            }
        )

    if deployment_favorable_count == 0:
        deployment_phrase = "No row is deployment-favorable under the median-burden screening rule."
    elif deployment_favorable_count == n:
        deployment_phrase = "All rows are deployment-favorable under the median-burden screening rule."
    else:
        deployment_phrase = (
            f"{deployment_favorable_count} of {n} rows are deployment-favorable "
            "under the median-burden screening rule."
        )

    if len(band_summary) == 2:
        c_cap = band_summary["C"]["total_bayesian_capacity_Gbps"]
        l_cap = band_summary["L"]["total_bayesian_capacity_Gbps"]
        if c_cap > l_cap:
            band_phrase = "C-band provides higher Bayesian-equalized capacity in the available rows."
        elif l_cap > c_cap:
            band_phrase = "L-band provides higher Bayesian-equalized capacity in the available rows."
        else:
            band_phrase = "C-band and L-band provide equal Bayesian-equalized capacity in the available rows."
    elif len(band_summary) == 1:
        band_phrase = f"Only {next(iter(band_summary.keys()))}-band rows are present."
    else:
        band_phrase = "No band summary is available."

    if float(total_capacity_gain.item()) > 0:
        gain_phrase = "The Bayesian CNN-BiLSTM equalizer improves total capacity relative to the no-NLC baseline."
    elif float(total_capacity_gain.item()) < 0:
        gain_phrase = "The Bayesian CNN-BiLSTM equalizer reduces total capacity relative to the no-NLC baseline."
    else:
        gain_phrase = "The Bayesian CNN-BiLSTM equalizer has zero net total capacity change relative to the no-NLC baseline."

    engineering_conclusion = (
        f"{gain_phrase} "
        f"Total Bayesian-equalized capacity is {_finite_float(total_bayesian_capacity.item()):.6g} Gbps, "
        f"with {_finite_float(total_capacity_gain.item()):.6g} Gbps total gain over the baseline. "
        f"{usable_count} rows are usable, {marginal_count} are marginal, and {unusable_count} are unusable. "
        f"{band_phrase} "
        f"The weakest bottleneck by NGMI is sample {sample_ids[lowest_ngmi_idx]}, "
        f"while the highest BER is sample {sample_ids[highest_ber_idx]}. "
        f"{deployment_phrase}"
    )

    output = {
        "sample_ids": [str(x) for x in sample_ids],
        "global_summary": {
            "row_count": int(n),
            "total_bayesian_capacity_Gbps": _finite_float(total_bayesian_capacity.item()),
            "total_no_nlc_capacity_Gbps": _finite_float(total_no_nlc_capacity.item()),
            "total_capacity_gain_Gbps": _finite_float(total_capacity_gain.item()),
            "mean_capacity_gain_Gbps": _finite_float(torch.mean(capacity_gain).item()),
            "mean_capacity_gain_pct": _finite_float(torch.mean(_tensor(data["capacity_gain_pct"])).item()),
            "mean_ber_reduction": _finite_float(torch.mean(ber_reduction).item()),
            "mean_gmi_gain": _finite_float(torch.mean(gmi_gain).item()),
            "mean_ngmi_gain": _finite_float(torch.mean(ngmi_gain).item()),
            "usable_channel_count": int(usable_count),
            "marginal_channel_count": int(marginal_count),
            "unusable_channel_count": int(unusable_count),
            "deployment_favorable_count": int(deployment_favorable_count),
            "deployment_favorable_fraction": _finite_float(deployment_favorable_count / float(n)),
        },
        "band_summary": band_summary,
        "per_row_report": per_row_report,
        "bottleneck_analysis": {
            "highest_capacity_gain_sample_id": str(sample_ids[highest_capacity_gain_idx]),
            "lowest_capacity_gain_sample_id": str(sample_ids[lowest_capacity_gain_idx]),
            "lowest_ngmi_sample_id": str(sample_ids[lowest_ngmi_idx]),
            "highest_ber_sample_id": str(sample_ids[highest_ber_idx]),
            "highest_uncertainty_sample_id": str(sample_ids[highest_uncertainty_idx]),
            "highest_deployment_burden_sample_id": str(sample_ids[highest_deployment_burden_idx]),
            "best_deployment_sample_id": str(sample_ids[best_deployment_idx]),
            "worst_deployment_sample_id": str(sample_ids[worst_deployment_idx]),
        },
        "engineering_conclusion": engineering_conclusion,
    }

    output = _to_builtin(output)

    output_file = Path(output_path)
    output_file.parent.mkdir(parents=True, exist_ok=True)
    output_file.write_text(json.dumps(output, indent=2), encoding="utf-8")

    return output
"""

In [ ]:
test_code = r"""
from pathlib import Path
import importlib.util
import io
import json
import math
import tempfile

import numpy as np
import pandas as pd
import pytest


DATA_DIR = Path("/workspace")
FUNCTION_FILE = DATA_DIR / "solve_cl_band_equalizer_engineering_audit.py"


REQUIRED_COLUMNS = [
    "sample_id",
    "channel_index",
    "channel_in_band",
    "band",
    "wavelength_nm",
    "distance_km",
    "loop_count",
    "symbol_rate_GBd",
    "modulation",
    "shaping_entropy_bits",
    "window_length",
    "launch_power_dBm",
    "osnr_dB",
    "srs_tilt_dB",
    "nonlinear_phase_proxy",
    "isi_memory_proxy",
    "amplitude_kurtosis_proxy",
    "center_rx_I",
    "center_rx_Q",
    "pre_context_I",
    "pre_context_Q",
    "post_context_I",
    "post_context_Q",
    "window_power_mean",
    "window_power_std",
    "window_iq_corr",
    "lag1_autocorr_I",
    "lag1_autocorr_Q",
    "target_residual_I",
    "target_residual_Q",
    "target_log_variance",
    "uncertainty_score",
    "ber_wo_nlc",
    "ber_bayesian_cnn_bilstm",
    "gmi_wo_nlc_bits_per_symbol",
    "gmi_bayesian_bits_per_symbol",
    "ngmi_wo_nlc",
    "ngmi_bayesian",
    "capacity_wo_nlc_Gbps",
    "capacity_bayesian_Gbps",
    "capacity_gain_pct",
    "epochs_bayesian_cnn_bilstm",
    "train_split",
]


GLOBAL_SUMMARY_KEYS = {
    "row_count",
    "total_bayesian_capacity_Gbps",
    "total_no_nlc_capacity_Gbps",
    "total_capacity_gain_Gbps",
    "mean_capacity_gain_Gbps",
    "mean_capacity_gain_pct",
    "mean_ber_reduction",
    "mean_gmi_gain",
    "mean_ngmi_gain",
    "usable_channel_count",
    "marginal_channel_count",
    "unusable_channel_count",
    "deployment_favorable_count",
    "deployment_favorable_fraction",
}


BAND_SUMMARY_KEYS = {
    "row_count",
    "total_bayesian_capacity_Gbps",
    "total_capacity_gain_Gbps",
    "mean_ber_reduction",
    "mean_gmi_gain",
    "mean_ngmi_gain",
    "mean_deployment_burden",
    "mean_equalizer_benefit",
    "deployment_favorable_fraction",
}


PER_ROW_KEYS = {
    "sample_id",
    "train_split",
    "band",
    "channel_index",
    "channel_in_band",
    "wavelength_nm",
    "ber_reduction",
    "gmi_gain",
    "ngmi_gain",
    "capacity_gain_Gbps",
    "residual_magnitude",
    "uncertainty_score",
    "quality_label",
    "deployment_burden",
    "equalizer_benefit",
    "deployment_favorable",
}


BOTTLENECK_KEYS = {
    "highest_capacity_gain_sample_id",
    "lowest_capacity_gain_sample_id",
    "lowest_ngmi_sample_id",
    "highest_ber_sample_id",
    "highest_uncertainty_sample_id",
    "highest_deployment_burden_sample_id",
    "best_deployment_sample_id",
    "worst_deployment_sample_id",
}


def load_function():
    if not FUNCTION_FILE.exists():
        raise FileNotFoundError(f"{FUNCTION_FILE} does not exist")

    spec = importlib.util.spec_from_file_location(
        "solve_cl_band_equalizer_engineering_audit",
        FUNCTION_FILE,
    )
    if spec is None or spec.loader is None:
        raise ImportError(f"Unable to load module from {FUNCTION_FILE}")

    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module.solve_cl_band_equalizer_engineering_audit


@pytest.fixture(scope="module")
def solve_cl_band_equalizer_engineering_audit():
    return load_function()


def _write_csv(path, text):
    Path(path).write_text(text, encoding="utf-8")


def _run_model(solve_cl_band_equalizer_engineering_audit, csv_text):
    with tempfile.TemporaryDirectory() as tmpdir:
        input_path = Path(tmpdir) / "cnn_bilstm_equalizer.csv"
        output_path = Path(tmpdir) / "output.json"

        _write_csv(input_path, csv_text)

        returned = solve_cl_band_equalizer_engineering_audit(
            input_path=str(input_path),
            output_path=str(output_path),
        )

        if not output_path.exists():
            raise AssertionError("output.json was not written")

        written = json.loads(output_path.read_text(encoding="utf-8"))
        return returned, written


def _run_model_nested_output(solve_cl_band_equalizer_engineering_audit, csv_text):
    with tempfile.TemporaryDirectory() as tmpdir:
        input_path = Path(tmpdir) / "cnn_bilstm_equalizer.csv"
        output_path = Path(tmpdir) / "nested" / "output.json"

        _write_csv(input_path, csv_text)

        returned = solve_cl_band_equalizer_engineering_audit(
            input_path=str(input_path),
            output_path=str(output_path),
        )

        if not output_path.exists():
            raise AssertionError("nested output.json was not written")

        written = json.loads(output_path.read_text(encoding="utf-8"))
        return returned, written


def _assert_invalid_input_rejected(fn):
    try:
        fn()
    except Exception:
        return
    raise AssertionError("Invalid input was not rejected")


def _assert_close(actual, expected, tol=1e-7):
    if abs(float(actual) - float(expected)) > tol:
        raise AssertionError(f"{actual} != {expected} within tol={tol}")


def _is_json_number(x):
    return (
        isinstance(x, (int, float))
        and not isinstance(x, bool)
        and math.isfinite(float(x))
    )


def _all_finite(obj):
    if isinstance(obj, dict):
        return all(_all_finite(v) for v in obj.values())
    if isinstance(obj, list):
        return all(_all_finite(v) for v in obj)
    if isinstance(obj, str) or isinstance(obj, bool) or obj is None:
        return True
    return _is_json_number(obj)


def _parse_clean_reference(csv_text):
    text = csv_text
    if text.startswith("\ufeff"):
        text = text[1:]
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    df = pd.read_csv(io.StringIO(text), dtype=str)
    df.columns = [str(c).strip() for c in df.columns]

    for col in df.columns:
        df[col] = df[col].map(lambda x: x.strip() if isinstance(x, str) else x)

    empty_mask = df.apply(
        lambda col: col.map(lambda x: isinstance(x, str) and x.strip() == "")
    )
    df = df.mask(empty_mask, np.nan)
    df = df.dropna(axis=0, how="all").dropna(axis=1, how="all")

    df.columns = [str(c).strip() for c in df.columns]
    for col in df.columns:
        df[col] = df[col].map(lambda x: x.strip() if isinstance(x, str) else x)

    df = df[REQUIRED_COLUMNS].copy()

    for col in REQUIRED_COLUMNS:
        if col not in {"sample_id", "band", "modulation", "train_split"}:
            df[col] = pd.to_numeric(df[col], errors="raise")

    return df


def _mutate_cell(csv_text, sample_id, column, value):
    df = pd.read_csv(io.StringIO(csv_text), dtype=str)
    if column not in df.columns:
        raise AssertionError(f"Column not found: {column}")
    mask = df["sample_id"].astype(str).str.strip() == sample_id
    if not mask.any():
        raise AssertionError(f"sample_id not found: {sample_id}")
    df.loc[mask, column] = str(value)
    return df.to_csv(index=False)


def _base_valid_csv():
    header = ",".join(REQUIRED_COLUMNS)
    rows = [
        [
            "s1", 1, 1, "C", 1524.50, 1512, 6, 96, "PCS-64-QAM", 5.5, 65,
            22.5, 20.5, -0.4, 0.08, 0.11, 2.90,
            0.12, -0.06, 0.05, -0.02, 0.10, -0.04,
            1.10, 0.14, 0.08, 0.12, 0.10,
            0.020, -0.010, -3.50, 0.18,
            0.030, 0.018, 5.10, 5.37, 0.86, 0.905,
            489.6, 515.5, 5.29, 18, "train",
        ],
        [
            "s2", 2, 2, "C", 1530.10, 1512, 6, 96, "PCS-64-QAM", 5.5, 65,
            22.8, 19.7, -0.6, 0.10, 0.13, 3.00,
            0.14, -0.05, 0.07, -0.01, 0.12, -0.03,
            1.24, 0.19, 0.12, 0.18, 0.15,
            0.024, -0.013, -3.20, 0.22,
            0.036, 0.021, 5.02, 5.31, 0.845, 0.894,
            481.9, 509.8, 5.79, 19, "validation",
        ],
        [
            "s3", 3, 3, "C", 1545.70, 1512, 6, 96, "PCS-64-QAM", 5.5, 65,
            23.0, 18.9, -0.8, 0.13, 0.16, 3.12,
            0.16, -0.03, 0.09, 0.00, 0.15, -0.02,
            1.38, 0.24, 0.16, 0.22, 0.20,
            0.030, -0.017, -2.90, 0.27,
            0.044, 0.026, 4.90, 5.20, 0.825, 0.878,
            470.4, 499.2, 6.12, 21, "test",
        ],
        [
            "s4", 4, 1, "L", 1578.20, 1512, 6, 96, "PCS-64-QAM", 5.5, 65,
            21.4, 18.2, -1.4, 0.17, 0.21, 3.28,
            0.18, 0.02, 0.12, 0.03, 0.17, 0.04,
            1.55, 0.30, 0.21, 0.28, 0.25,
            0.038, -0.020, -2.55, 0.34,
            0.058, 0.034, 4.72, 5.04, 0.798, 0.855,
            453.1, 483.8, 6.78, 22, "train",
        ],
        [
            "s5", 5, 2, "L", 1601.30, 1512, 6, 96, "PCS-64-QAM", 5.5, 65,
            21.1, 17.6, -1.7, 0.20, 0.25, 3.39,
            0.21, 0.04, 0.15, 0.05, 0.20, 0.06,
            1.72, 0.35, 0.27, 0.34, 0.30,
            0.045, -0.026, -2.25, 0.41,
            0.070, 0.041, 4.58, 4.94, 0.774, 0.840,
            439.7, 474.2, 7.85, 23, "validation",
        ],
        [
            "s6", 6, 3, "L", 1620.40, 1512, 6, 96, "PCS-64-QAM", 5.0, 65,
            20.8, 17.0, -1.9, 0.23, 0.28, 3.55,
            0.24, 0.07, 0.17, 0.08, 0.23, 0.10,
            1.91, 0.42, 0.31, 0.40, 0.36,
            0.054, -0.031, -1.95, 0.50,
            0.086, 0.050, 4.38, 4.79, 0.740, 0.815,
            420.5, 459.8, 9.35, 25, "test",
        ],
        [
            "s7", 7, 4, "L", 1630.00, 1512, 6, 96, "PCS-64-QAM", 5.0, 65,
            20.5, 15.2, -2.2, 0.30, 0.35, 3.70,
            0.30, 0.10, 0.20, 0.12, 0.27, 0.15,
            2.05, 0.50, 0.36, 0.46, 0.42,
            0.065, -0.038, -1.60, 0.68,
            0.140, 0.115, 4.05, 4.21, 0.650, 0.680,
            388.8, 404.2, 3.96, 30, "test",
        ],
    ]
    return header + "\n" + "\n".join(",".join(map(str, r)) for r in rows) + "\n"


def _single_band_valid_csv():
    lines = _base_valid_csv().strip().splitlines()
    return "\n".join([lines[0]] + [line for line in lines[1:] if ",C," in line]) + "\n"


def _constant_feature_valid_csv():
    df = pd.read_csv(io.StringIO(_base_valid_csv()), dtype=str)
    constant_cols = [
        "window_length",
        "epochs_bayesian_cnn_bilstm",
        "symbol_rate_GBd",
        "isi_memory_proxy",
        "nonlinear_phase_proxy",
        "window_power_std",
        "uncertainty_score",
        "target_log_variance",
        "capacity_gain_pct",
    ]
    for col in constant_cols:
        df[col] = df[col].iloc[0]
    return df.to_csv(index=False)


def _expected_arrays(csv_text):
    df = _parse_clean_reference(csv_text)

    ber_reduction = df["ber_wo_nlc"].to_numpy(float) - df["ber_bayesian_cnn_bilstm"].to_numpy(float)
    gmi_gain = df["gmi_bayesian_bits_per_symbol"].to_numpy(float) - df["gmi_wo_nlc_bits_per_symbol"].to_numpy(float)
    ngmi_gain = df["ngmi_bayesian"].to_numpy(float) - df["ngmi_wo_nlc"].to_numpy(float)
    capacity_gain = df["capacity_bayesian_Gbps"].to_numpy(float) - df["capacity_wo_nlc_Gbps"].to_numpy(float)
    residual = np.sqrt(
        df["target_residual_I"].to_numpy(float) ** 2
        + df["target_residual_Q"].to_numpy(float) ** 2
    )

    quality = []
    for _, row in df.iterrows():
        ber = float(row["ber_bayesian_cnn_bilstm"])
        ngmi = float(row["ngmi_bayesian"])
        if ber <= 0.05 and ngmi >= 0.80:
            quality.append("usable")
        elif ber > 0.10 or ngmi < 0.70:
            quality.append("unusable")
        else:
            quality.append("marginal")

    return {
        "df": df,
        "sample_ids": df["sample_id"].tolist(),
        "ber_reduction": ber_reduction,
        "gmi_gain": gmi_gain,
        "ngmi_gain": ngmi_gain,
        "capacity_gain": capacity_gain,
        "residual": residual,
        "quality": quality,
    }


def _minmax_np(x):
    x = np.asarray(x, dtype=float)
    lo = float(np.min(x))
    hi = float(np.max(x))
    if abs(hi - lo) <= 1e-12:
        return np.zeros_like(x, dtype=float)
    return (x - lo) / (hi - lo)


def _expected_deployment(csv_text):
    e = _expected_arrays(csv_text)
    df = e["df"]

    burden_terms = np.vstack(
        [
            _minmax_np(df["window_length"].to_numpy(float)),
            _minmax_np(df["epochs_bayesian_cnn_bilstm"].to_numpy(float)),
            _minmax_np(df["symbol_rate_GBd"].to_numpy(float)),
            _minmax_np(df["isi_memory_proxy"].to_numpy(float)),
            _minmax_np(df["nonlinear_phase_proxy"].to_numpy(float)),
            _minmax_np(df["window_power_std"].to_numpy(float)),
            _minmax_np(df["uncertainty_score"].to_numpy(float)),
            _minmax_np(df["target_log_variance"].to_numpy(float)),
        ]
    ).T
    burden = np.mean(burden_terms, axis=1)

    benefit_terms = np.vstack(
        [
            _minmax_np(np.maximum(e["ber_reduction"], 0.0)),
            _minmax_np(np.maximum(e["gmi_gain"], 0.0)),
            _minmax_np(np.maximum(e["ngmi_gain"], 0.0)),
            _minmax_np(np.maximum(e["capacity_gain"], 0.0)),
            _minmax_np(np.maximum(df["capacity_gain_pct"].to_numpy(float), 0.0)),
        ]
    ).T
    benefit = np.mean(benefit_terms, axis=1)

    median_burden = float(np.median(burden))
    favorable = (
        (e["capacity_gain"] > 0.0)
        & (e["gmi_gain"] > 0.0)
        & (e["ngmi_gain"] > 0.0)
        & (burden <= median_burden)
    )
    deployment_score = benefit - burden

    return burden, benefit, favorable, deployment_score


def _assert_schema(out):
    expected_top = {
        "sample_ids",
        "global_summary",
        "band_summary",
        "per_row_report",
        "bottleneck_analysis",
        "engineering_conclusion",
    }
    if set(out.keys()) != expected_top:
        raise AssertionError("Top-level schema mismatch")

    if set(out["global_summary"].keys()) != GLOBAL_SUMMARY_KEYS:
        raise AssertionError("global_summary schema mismatch")

    if set(out["bottleneck_analysis"].keys()) != BOTTLENECK_KEYS:
        raise AssertionError("bottleneck_analysis schema mismatch")

    for row in out["per_row_report"]:
        if set(row.keys()) != PER_ROW_KEYS:
            raise AssertionError("per_row_report schema mismatch")

    for summary in out["band_summary"].values():
        if set(summary.keys()) != BAND_SUMMARY_KEYS:
            raise AssertionError("band_summary schema mismatch")


def test_case_1(solve_cl_band_equalizer_engineering_audit):
    returned, out = _run_model(solve_cl_band_equalizer_engineering_audit, _base_valid_csv())

    if returned != out:
        raise AssertionError("Returned object and written JSON differ")

    _assert_schema(out)

    if not _all_finite(out):
        raise AssertionError("Output contains non-finite values")


def test_case_2(solve_cl_band_equalizer_engineering_audit):
    csv = _base_valid_csv()
    _, out = _run_model(solve_cl_band_equalizer_engineering_audit, csv)
    e = _expected_arrays(csv)

    if out["sample_ids"] != e["sample_ids"]:
        raise AssertionError("sample_ids must preserve cleaned row order")

    if [row["sample_id"] for row in out["per_row_report"]] != e["sample_ids"]:
        raise AssertionError("per_row_report must preserve cleaned row order")


def test_case_3(solve_cl_band_equalizer_engineering_audit):
    header = ",".join([f" {c} " for c in REQUIRED_COLUMNS] + [" empty_extra "])

    row_a_values = [
        " a ", 1, 1, " C ", 1524.5, 1512, 6, 96, " PCS-64-QAM ", 5.5, 65,
        22.5, 20.5, -0.4, 0.08, 0.11, 2.9,
        0.12, -0.06, 0.05, -0.02, 0.10, -0.04,
        1.10, 0.14, 0.08, 0.12, 0.10,
        0.020, -0.010, -3.5, 0.18,
        0.030, 0.018, 5.10, 5.37, 0.86, 0.905,
        489.6, 515.5, 5.29, 18, " train ", ""
    ]

    row_b_values = [
        " b ", 2, 1, " L ", 1578.2, 1512, 6, 96, " PCS-64-QAM ", 5.5, 65,
        21.4, 18.2, -1.4, 0.17, 0.21, 3.28,
        0.18, 0.02, 0.12, 0.03, 0.17, 0.04,
        1.55, 0.30, 0.21, 0.28, 0.25,
        0.038, -0.020, -2.55, 0.34,
        0.058, 0.034, 4.72, 5.04, 0.798, 0.855,
        453.1, 483.8, 6.78, 22, " test ", ""
    ]

    empty_row = [""] * (len(REQUIRED_COLUMNS) + 1)

    csv = (
        "\ufeff" + header + "\r\n"
        + ",".join(map(str, row_a_values)) + "\r\n"
        + ",".join(empty_row) + "\r\n"
        + ",".join(map(str, row_b_values)) + "\n"
    )

    _, out = _run_model(solve_cl_band_equalizer_engineering_audit, csv)

    if out["sample_ids"] != ["a", "b"]:
        raise AssertionError("BOM/whitespace/empty-row/line-ending cleaning failed")

    if [row["train_split"] for row in out["per_row_report"]] != ["train", "test"]:
        raise AssertionError("train_split must be preserved after trimming")


def test_case_4(solve_cl_band_equalizer_engineering_audit):
    returned, written = _run_model_nested_output(
        solve_cl_band_equalizer_engineering_audit,
        _base_valid_csv(),
    )

    if returned != written:
        raise AssertionError("Returned object and nested written JSON differ")

    _assert_schema(written)


def test_case_5(solve_cl_band_equalizer_engineering_audit):
    csv = _base_valid_csv()
    _, out = _run_model(solve_cl_band_equalizer_engineering_audit, csv)
    e = _expected_arrays(csv)

    for i, row in enumerate(out["per_row_report"]):
        _assert_close(row["ber_reduction"], e["ber_reduction"][i])
        _assert_close(row["gmi_gain"], e["gmi_gain"][i])
        _assert_close(row["ngmi_gain"], e["ngmi_gain"][i])
        _assert_close(row["capacity_gain_Gbps"], e["capacity_gain"][i])
        _assert_close(row["residual_magnitude"], e["residual"][i])


def test_case_6(solve_cl_band_equalizer_engineering_audit):
    csv = _base_valid_csv()
    _, out = _run_model(solve_cl_band_equalizer_engineering_audit, csv)
    e = _expected_arrays(csv)

    labels = [row["quality_label"] for row in out["per_row_report"]]
    if labels != e["quality"]:
        raise AssertionError("quality_label values do not match required thresholds")

    if not set(labels).issubset({"usable", "marginal", "unusable"}):
        raise AssertionError("quality_label must be one of usable, marginal, or unusable")


def test_case_7(solve_cl_band_equalizer_engineering_audit):
    csv = _base_valid_csv()
    _, out = _run_model(solve_cl_band_equalizer_engineering_audit, csv)
    e = _expected_arrays(csv)

    gs = out["global_summary"]
    _assert_close(gs["total_bayesian_capacity_Gbps"], e["df"]["capacity_bayesian_Gbps"].sum())
    _assert_close(gs["total_no_nlc_capacity_Gbps"], e["df"]["capacity_wo_nlc_Gbps"].sum())
    _assert_close(gs["total_capacity_gain_Gbps"], np.sum(e["capacity_gain"]))
    _assert_close(gs["mean_capacity_gain_Gbps"], np.mean(e["capacity_gain"]))
    _assert_close(gs["mean_capacity_gain_pct"], e["df"]["capacity_gain_pct"].mean())
    _assert_close(gs["mean_ber_reduction"], np.mean(e["ber_reduction"]))
    _assert_close(gs["mean_gmi_gain"], np.mean(e["gmi_gain"]))
    _assert_close(gs["mean_ngmi_gain"], np.mean(e["ngmi_gain"]))


def test_case_8(solve_cl_band_equalizer_engineering_audit):
    csv = _base_valid_csv()
    _, out = _run_model(solve_cl_band_equalizer_engineering_audit, csv)
    e = _expected_arrays(csv)

    gs = out["global_summary"]
    assert gs["row_count"] == len(e["df"])
    assert gs["usable_channel_count"] == e["quality"].count("usable")
    assert gs["marginal_channel_count"] == e["quality"].count("marginal")
    assert gs["unusable_channel_count"] == e["quality"].count("unusable")
    assert (
        gs["usable_channel_count"]
        + gs["marginal_channel_count"]
        + gs["unusable_channel_count"]
        == gs["row_count"]
    )


def test_case_9(solve_cl_band_equalizer_engineering_audit):
    csv = _base_valid_csv()
    _, out = _run_model(solve_cl_band_equalizer_engineering_audit, csv)
    e = _expected_arrays(csv)
    burden, benefit, favorable, _ = _expected_deployment(csv)

    rows = out["per_row_report"]
    for i, row in enumerate(rows):
        _assert_close(row["deployment_burden"], burden[i])
        _assert_close(row["equalizer_benefit"], benefit[i])
        if row["deployment_favorable"] != bool(favorable[i]):
            raise AssertionError("deployment_favorable does not match required rule")

    assert out["global_summary"]["deployment_favorable_count"] == int(np.sum(favorable))
    _assert_close(
        out["global_summary"]["deployment_favorable_fraction"],
        float(np.mean(favorable)),
    )


def test_case_10(solve_cl_band_equalizer_engineering_audit):
    csv = _constant_feature_valid_csv()
    _, out = _run_model(solve_cl_band_equalizer_engineering_audit, csv)

    for row in out["per_row_report"]:
        if row["deployment_burden"] < -1e-12:
            raise AssertionError("deployment_burden must be nonnegative")
        if row["equalizer_benefit"] < -1e-12:
            raise AssertionError("equalizer_benefit must be nonnegative")
        if not math.isfinite(row["deployment_burden"]):
            raise AssertionError("deployment_burden must remain finite for constant features")
        if not math.isfinite(row["equalizer_benefit"]):
            raise AssertionError("equalizer_benefit must remain finite for constant features")


def test_case_11(solve_cl_band_equalizer_engineering_audit):
    csv = _base_valid_csv()
    _, out = _run_model(solve_cl_band_equalizer_engineering_audit, csv)
    e = _expected_arrays(csv)
    burden, benefit, favorable, _ = _expected_deployment(csv)

    expected_bands = set(e["df"]["band"])
    if set(out["band_summary"].keys()) != expected_bands:
        raise AssertionError("band_summary must contain all and only present bands")

    for band in expected_bands:
        mask = e["df"]["band"].to_numpy() == band
        summary = out["band_summary"][band]
        assert summary["row_count"] == int(np.sum(mask))
        _assert_close(summary["total_bayesian_capacity_Gbps"], e["df"]["capacity_bayesian_Gbps"].to_numpy(float)[mask].sum())
        _assert_close(summary["total_capacity_gain_Gbps"], e["capacity_gain"][mask].sum())
        _assert_close(summary["mean_ber_reduction"], e["ber_reduction"][mask].mean())
        _assert_close(summary["mean_gmi_gain"], e["gmi_gain"][mask].mean())
        _assert_close(summary["mean_ngmi_gain"], e["ngmi_gain"][mask].mean())
        _assert_close(summary["mean_deployment_burden"], burden[mask].mean())
        _assert_close(summary["mean_equalizer_benefit"], benefit[mask].mean())
        _assert_close(summary["deployment_favorable_fraction"], favorable[mask].mean())


def test_case_12(solve_cl_band_equalizer_engineering_audit):
    csv = _single_band_valid_csv()
    _, out = _run_model(solve_cl_band_equalizer_engineering_audit, csv)

    if set(out["band_summary"].keys()) != {"C"}:
        raise AssertionError("Single-band input should summarize only the present band")

    if len(out["per_row_report"]) != 3:
        raise AssertionError("Single-band input should still use all cleaned rows")


def test_case_13(solve_cl_band_equalizer_engineering_audit):
    csv = _base_valid_csv()
    _, out = _run_model(solve_cl_band_equalizer_engineering_audit, csv)
    e = _expected_arrays(csv)
    burden, benefit, favorable, deployment_score = _expected_deployment(csv)

    ba = out["bottleneck_analysis"]

    assert ba["highest_capacity_gain_sample_id"] == e["sample_ids"][int(np.argmax(e["capacity_gain"]))]
    assert ba["lowest_capacity_gain_sample_id"] == e["sample_ids"][int(np.argmin(e["capacity_gain"]))]
    assert ba["lowest_ngmi_sample_id"] == e["sample_ids"][int(np.argmin(e["df"]["ngmi_bayesian"].to_numpy(float)))]
    assert ba["highest_ber_sample_id"] == e["sample_ids"][int(np.argmax(e["df"]["ber_bayesian_cnn_bilstm"].to_numpy(float)))]
    assert ba["highest_uncertainty_sample_id"] == e["sample_ids"][int(np.argmax(e["df"]["uncertainty_score"].to_numpy(float)))]
    assert ba["highest_deployment_burden_sample_id"] == e["sample_ids"][int(np.argmax(burden))]
    assert ba["best_deployment_sample_id"] == e["sample_ids"][int(np.argmax(deployment_score))]
    assert ba["worst_deployment_sample_id"] == e["sample_ids"][int(np.argmin(deployment_score))]


def test_case_14(solve_cl_band_equalizer_engineering_audit):
    csv = _base_valid_csv()
    csv = _mutate_cell(csv, "s1", "capacity_bayesian_Gbps", 600.0)
    csv = _mutate_cell(csv, "s2", "capacity_bayesian_Gbps", 600.0)
    csv = _mutate_cell(csv, "s1", "capacity_wo_nlc_Gbps", 500.0)
    csv = _mutate_cell(csv, "s2", "capacity_wo_nlc_Gbps", 500.0)

    _, out = _run_model(solve_cl_band_equalizer_engineering_audit, csv)

    if out["bottleneck_analysis"]["highest_capacity_gain_sample_id"] != "s1":
        raise AssertionError("Ties for best/worst identifiers must use earliest cleaned row order")


def test_case_15(solve_cl_band_equalizer_engineering_audit):
    csv = _base_valid_csv()
    _, out = _run_model(solve_cl_band_equalizer_engineering_audit, csv)

    conclusion = out["engineering_conclusion"]
    if not isinstance(conclusion, str) or not conclusion.strip():
        raise AssertionError("engineering_conclusion must be non-empty text")

    required_mentions = [
        str(out["global_summary"]["usable_channel_count"]),
        out["bottleneck_analysis"]["lowest_ngmi_sample_id"],
        out["bottleneck_analysis"]["highest_ber_sample_id"],
    ]
    for text in required_mentions:
        if text not in conclusion:
            raise AssertionError("engineering_conclusion should be based on computed outputs")


def test_case_16(solve_cl_band_equalizer_engineering_audit):
    csv = _base_valid_csv()
    _, out1 = _run_model(solve_cl_band_equalizer_engineering_audit, csv)
    _, out2 = _run_model(solve_cl_band_equalizer_engineering_audit, csv)

    if out1 != out2:
        raise AssertionError("Solution must be deterministic for the same input")


def test_case_17(solve_cl_band_equalizer_engineering_audit):
    csv_base = _base_valid_csv()
    csv_changed = _mutate_cell(csv_base, "s1", "capacity_bayesian_Gbps", 600.0)

    _, out_base = _run_model(solve_cl_band_equalizer_engineering_audit, csv_base)
    _, out_changed = _run_model(solve_cl_band_equalizer_engineering_audit, csv_changed)

    if out_base["global_summary"]["total_bayesian_capacity_Gbps"] == out_changed["global_summary"]["total_bayesian_capacity_Gbps"]:
        raise AssertionError("Capacity totals must respond to capacity_bayesian_Gbps changes")


def test_case_18(solve_cl_band_equalizer_engineering_audit):
    csv_base = _base_valid_csv()
    csv_changed = _mutate_cell(csv_base, "s1", "ber_bayesian_cnn_bilstm", 0.004)

    _, out_base = _run_model(solve_cl_band_equalizer_engineering_audit, csv_base)
    _, out_changed = _run_model(solve_cl_band_equalizer_engineering_audit, csv_changed)

    if out_base["per_row_report"][0]["ber_reduction"] == out_changed["per_row_report"][0]["ber_reduction"]:
        raise AssertionError("BER reduction must respond to Bayesian BER changes")


def test_case_19(solve_cl_band_equalizer_engineering_audit):
    csv_base = _base_valid_csv()
    csv_changed = _mutate_cell(csv_base, "s1", "window_length", 129)

    _, out_base = _run_model(solve_cl_band_equalizer_engineering_audit, csv_base)
    _, out_changed = _run_model(solve_cl_band_equalizer_engineering_audit, csv_changed)

    burdens_base = [row["deployment_burden"] for row in out_base["per_row_report"]]
    burdens_changed = [row["deployment_burden"] for row in out_changed["per_row_report"]]

    if burdens_base == burdens_changed:
        raise AssertionError("Deployment burden must respond to required burden features")


def test_case_20(solve_cl_band_equalizer_engineering_audit):
    csv_base = _base_valid_csv()
    csv_changed = _mutate_cell(csv_base, "s1", "capacity_gain_pct", 20.0)

    _, out_base = _run_model(solve_cl_band_equalizer_engineering_audit, csv_base)
    _, out_changed = _run_model(solve_cl_band_equalizer_engineering_audit, csv_changed)

    benefits_base = [row["equalizer_benefit"] for row in out_base["per_row_report"]]
    benefits_changed = [row["equalizer_benefit"] for row in out_changed["per_row_report"]]

    if benefits_base == benefits_changed:
        raise AssertionError("Equalizer benefit must respond to required benefit features")


def test_case_21(solve_cl_band_equalizer_engineering_audit):
    csv = ",".join(REQUIRED_COLUMNS[:8]) + "\n"
    csv += "s1,1,1,C,1524.5,1512,6,96\n"
    _assert_invalid_input_rejected(lambda: _run_model(solve_cl_band_equalizer_engineering_audit, csv))


def test_case_22(solve_cl_band_equalizer_engineering_audit):
    csv = _mutate_cell(_base_valid_csv(), "s2", "sample_id", "s1")
    _assert_invalid_input_rejected(lambda: _run_model(solve_cl_band_equalizer_engineering_audit, csv))


def test_case_23(solve_cl_band_equalizer_engineering_audit):
    csv = _mutate_cell(_base_valid_csv(), "s2", "sample_id", "   ")
    _assert_invalid_input_rejected(lambda: _run_model(solve_cl_band_equalizer_engineering_audit, csv))


def test_case_24(solve_cl_band_equalizer_engineering_audit):
    csv = _mutate_cell(_base_valid_csv(), "s1", "band", "S")
    _assert_invalid_input_rejected(lambda: _run_model(solve_cl_band_equalizer_engineering_audit, csv))


def test_case_25(solve_cl_band_equalizer_engineering_audit):
    csv = _mutate_cell(_base_valid_csv(), "s1", "modulation", "   ")
    _assert_invalid_input_rejected(lambda: _run_model(solve_cl_band_equalizer_engineering_audit, csv))


def test_case_26(solve_cl_band_equalizer_engineering_audit):
    csv = _mutate_cell(_base_valid_csv(), "s1", "channel_index", "not_a_number")
    _assert_invalid_input_rejected(lambda: _run_model(solve_cl_band_equalizer_engineering_audit, csv))


def test_case_27(solve_cl_band_equalizer_engineering_audit):
    csv = _mutate_cell(_base_valid_csv(), "s1", "channel_index", "inf")
    _assert_invalid_input_rejected(lambda: _run_model(solve_cl_band_equalizer_engineering_audit, csv))


def test_case_28(solve_cl_band_equalizer_engineering_audit):
    csv = _mutate_cell(_base_valid_csv(), "s1", "channel_index", 0)
    _assert_invalid_input_rejected(lambda: _run_model(solve_cl_band_equalizer_engineering_audit, csv))


def test_case_29(solve_cl_band_equalizer_engineering_audit):
    csv = _mutate_cell(_base_valid_csv(), "s1", "distance_km", -1)
    _assert_invalid_input_rejected(lambda: _run_model(solve_cl_band_equalizer_engineering_audit, csv))


def test_case_30(solve_cl_band_equalizer_engineering_audit):
    csv = _mutate_cell(_base_valid_csv(), "s1", "ber_bayesian_cnn_bilstm", 1.3)
    _assert_invalid_input_rejected(lambda: _run_model(solve_cl_band_equalizer_engineering_audit, csv))


def test_case_31(solve_cl_band_equalizer_engineering_audit):
    csv = _mutate_cell(_base_valid_csv(), "s1", "ngmi_bayesian", -0.1)
    _assert_invalid_input_rejected(lambda: _run_model(solve_cl_band_equalizer_engineering_audit, csv))


def test_case_32(solve_cl_band_equalizer_engineering_audit):
    csv = _mutate_cell(_base_valid_csv(), "s1", "window_iq_corr", 1.2)
    _assert_invalid_input_rejected(lambda: _run_model(solve_cl_band_equalizer_engineering_audit, csv))


def test_case_33(solve_cl_band_equalizer_engineering_audit):
    csv = _mutate_cell(_base_valid_csv(), "s1", "lag1_autocorr_I", -1.2)
    _assert_invalid_input_rejected(lambda: _run_model(solve_cl_band_equalizer_engineering_audit, csv))


def test_case_34(solve_cl_band_equalizer_engineering_audit):
    csv = _mutate_cell(_base_valid_csv(), "s1", "window_length", 64)
    _assert_invalid_input_rejected(lambda: _run_model(solve_cl_band_equalizer_engineering_audit, csv))


def test_case_35(solve_cl_band_equalizer_engineering_audit):
    csv = _mutate_cell(_base_valid_csv(), "s1", "window_length", 65.5)
    _assert_invalid_input_rejected(lambda: _run_model(solve_cl_band_equalizer_engineering_audit, csv))


def test_case_36(solve_cl_band_equalizer_engineering_audit):
    csv = _mutate_cell(_base_valid_csv(), "s1", "epochs_bayesian_cnn_bilstm", 0)
    _assert_invalid_input_rejected(lambda: _run_model(solve_cl_band_equalizer_engineering_audit, csv))


def test_case_37(solve_cl_band_equalizer_engineering_audit):
    csv = _mutate_cell(_base_valid_csv(), "s1", "capacity_bayesian_Gbps", -2)
    _assert_invalid_input_rejected(lambda: _run_model(solve_cl_band_equalizer_engineering_audit, csv))


def test_case_38(solve_cl_band_equalizer_engineering_audit):
    csv = (
        ",".join(REQUIRED_COLUMNS + ["unused_empty"]) + "\n"
        "x1,1,1,C,1524.5,1512,6,96,PCS-64-QAM,5.5,65,22.5,20.5,-0.4,0.08,0.11,2.90,0.12,-0.06,0.05,-0.02,0.10,-0.04,1.10,0.14,0.08,0.12,0.10,0.020,-0.010,-3.5,0.18,0.030,0.018,5.10,5.37,0.86,0.905,489.6,515.5,5.29,18,train,\n"
        "x2,2,1,L,1578.2,1512,6,96,PCS-64-QAM,5.5,65,21.4,18.2,-1.4,0.17,0.21,3.28,0.18,0.02,0.12,0.03,0.17,0.04,1.55,0.30,0.21,0.28,0.25,0.038,-0.020,-2.55,0.34,0.058,0.034,4.72,5.04,0.798,0.855,453.1,483.8,6.78,22,test,\n"
    )
    _, out = _run_model(solve_cl_band_equalizer_engineering_audit, csv)

    if out["sample_ids"] != ["x1", "x2"]:
        raise AssertionError("Fully empty additional columns should be ignored after cleaning")
"""

In [ ]:
from pathlib import Path
import re
import subprocess
import sys

# 1. Write files (ALWAYS overwrite)
Path("/workspace").mkdir(parents=True, exist_ok=True)

Path("/workspace/solve_cl_band_equalizer_engineering_audit.py").write_text(
    solution_code,
    encoding="utf-8",
)

Path("/workspace/test_solve_cl_band_equalizer_engineering_audit.py").write_text(
    test_code,
    encoding="utf-8",
)

print("Files written.")

# 2. Verify test count
test_path = Path("/workspace/test_solve_cl_band_equalizer_engineering_audit.py")
test_text = test_path.read_text(encoding="utf-8")

tests = re.findall(
    r"^def\s+(test_case_\d+)\s*\(",
    test_text,
    flags=re.MULTILINE,
)

print(f"\nFound {len(tests)} test_case_* functions")
print(tests)

assert len(tests) == 38, "Not all 38 tests are present!"

# 3. Sanity-check task-specific filenames and function names
solution_path = Path("/workspace/solve_cl_band_equalizer_engineering_audit.py")

assert solution_path.exists(), "Solution file was not written."

solution_text = solution_path.read_text(encoding="utf-8")

assert "solve_cl_band_equalizer_engineering_audit" in solution_text, (
    "Solution file does not define the expected function name."
)

assert "solve_cl_band_equalizer_engineering_audit" in test_text, (
    "Test file does not target the expected function name."
)

old_names = [
    "solve_equalizer_deployment_efficiency_audit",
    "solve_equalizer_deployment_efficiency_audit.py",
    "solve_counterfactual_propagation_audit",
    "solve_counterfactual_propagation_audit.py",
    "solve_pcs_shannon_gap_audit",
    "solve_pcs_shannon_gap_audit.py",
    "solve_uncertainty_equalizer_audit",
    "solve_uncertainty_equalizer_audit.py",
    "solve_stopping_response_transport",
    "solve_stopping_response_transport.py",
    "solve_ion_track_response_field",
    "solve_ion_track_response_field.py",
]

for old_name in old_names:
    assert old_name not in test_text, (
        f"Old task reference is still present in the test file: {old_name}"
    )

# 4. Collect tests
# pytest discovers test_*.py files and test_* functions by default.
# --collect-only lists collected tests without executing them.
collect = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        str(test_path),
        "--collect-only",
        "-q",
    ],
    capture_output=True,
    text=True,
)

print("\nCOLLECT OUTPUT:\n")
print(collect.stdout)
print(collect.stderr)

if collect.returncode != 0:
    raise RuntimeError("Pytest collection failed.")

# 5. Run tests
run = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        str(test_path),
        "-q",
    ],
    capture_output=True,
    text=True,
)

print("\nTEST RUN OUTPUT:\n")
print(run.stdout)
print(run.stderr)

if run.returncode == 0:
    print("\nALL TESTS PASSED")
else:
    print("\nSOME TESTS FAILED")

Files written.

Found 38 test_case_* functions
['test_case_1', 'test_case_2', 'test_case_3', 'test_case_4', 'test_case_5', 'test_case_6', 'test_case_7', 'test_case_8', 'test_case_9', 'test_case_10', 'test_case_11', 'test_case_12', 'test_case_13', 'test_case_14', 'test_case_15', 'test_case_16', 'test_case_17', 'test_case_18', 'test_case_19', 'test_case_20', 'test_case_21', 'test_case_22', 'test_case_23', 'test_case_24', 'test_case_25', 'test_case_26', 'test_case_27', 'test_case_28', 'test_case_29', 'test_case_30', 'test_case_31', 'test_case_32', 'test_case_33', 'test_case_34', 'test_case_35', 'test_case_36', 'test_case_37', 'test_case_38']

COLLECT OUTPUT:

test_solve_cl_band_equalizer_engineering_audit.py::test_case_1
test_solve_cl_band_equalizer_engineering_audit.py::test_case_2
test_solve_cl_band_equalizer_engineering_audit.py::test_case_3
test_solve_cl_band_equalizer_engineering_audit.py::test_case_4
test_solve_cl_band_equalizer_engineering_audit.py::test_case_5
test_solve_cl_band_e